In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

if HF_TOKEN:
    login(HF_TOKEN)
    print("Logged in to Hugging Face!")
else:
    print("HF_TOKEN not found. Please add it in Colab secrets.")

In [ ]:
# Install dependencies
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes

#imports

In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import defaultdict
import os
import torch
import numpy as np
import subprocess
import sys
from datasets import load_dataset, Dataset
from transformers import (
AutoModelForCausalLM,
AutoTokenizer,
BitsAndBytesConfig,
TrainingArguments,
Trainer,
DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import warnings
warnings.filterwarnings('ignore')

# LOAD AND PREPARE DATASET WITH STREAMING

In [ ]:
import datasets
from datasets import load_dataset

# Load the dataset from Hugging Face Hub
print(f"\nUsing version: {datasets.__version__}\n")
# Specify a revision that uses a different data format (e.g., Parquet)
try:
    dataset = ds_hard = load_dataset("kellycyy/CulturalBench", "CulturalBench-Hard")#Skylion007/openwebtext", revision="refs/convert/parquet")
    print("Dataset loaded successfully using parquet revision.")
except Exception as e:
    print(f"Failed to load with parquet revision: {e}")
    # Fallback or alternative approach if parquet fails
    print("Attempting to load a different subset or version if available, or consider alternative datasets.")
    # Note: For openwebtext, the parquet revision is generally the recommended fix for this error.
    # If this still fails, the dataset itself might be temporarily unavailable or require a different approach.
    raise # Re-raise the exception if the fix doesn't work

#CONFIGURATION

In [ ]:
# Model configuration
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
OUTPUT_DIR = "./llama-3.2-3b-finetuned-CulturalBench"

# Dataset configuration - Using OpenWebText with streaming
DATASET_NAME = dataset
MAX_SEQ_LENGTH = 512  # Reduced for T4 GPU memory constraints
DATASET_SPLIT = "train" # wikitext has 'train', 'validation', 'test' splits

# Training configuration
PER_DEVICE_TRAIN_BATCH_SIZE = 1  # Small batch size for T4
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch size = 4
LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 1  # Set to 1 for faster completion; increase if needed
MAX_STEPS = 70  # Limit steps for demo; remove or increase for full training
WARMUP_STEPS = 10
LOGGING_STEPS = 10
SAVE_STEPS = 50

# LoRA configuration
LORA_R = 16  # Rank
LORA_ALPHA = 32  # Alpha parameter
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

#4-BIT QUANTIZATION CONFIGURATION

In [ ]:
print("Setting up 4-bit quantization configuration...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",  # NormalFloat 4-bit quantization
    bnb_4bit_compute_dtype=torch.bfloat16,  # Compute dtype for operations
    bnb_4bit_use_double_quant=True,  # Double quantization for memory efficiency
)

#TOKENIZER

In [ ]:
print(f"Loading model: {MODEL_NAME}")
print("This will download the model if not cached...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True
)

# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "right"  # Required for training

#Model Loading

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
    attn_implementation="eager"
)

#Prepare model for k-bit training

In [ ]:
model = prepare_model_for_kbit_training(model)

# Disable cache for training
model.config.use_cache = True
model.config.pretraining_tp = 1

print(f"Model loaded successfully!")
print(f"Model memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

#CONFIGURE LORA

In [ ]:
print("Configuring LoRA...")

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM"
)

# Get PEFT model
model = get_peft_model(model, lora_config)

#trainable parameters

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / all_params:.2f}%)")
print(f"All parameters: {all_params:,}")

# Tokenization function

In [ ]:
dataset.column_names

In [ ]:
type(dataset["test"]["prompt_question"])

In [ ]:
# Inspect data types of the columns
print("Inspecting data types of relevant columns:")
for col in ["prompt_question", "prompt_option", "answer"]:
    if col in dataset["test"].column_names:
        print(f"Column '{col}': {dataset['test'][col].features.dtype}")
    else:
        print(f"Column '{col}' not found in dataset.")

# Inspect a few examples to see the values
print("\nInspecting first 5 examples:")
for i in range(5):
    example = dataset["test"][i]
    print(f"Example {i}:")
    for col in ["prompt_question", "prompt_option", "answer"]:
        if col in example:
            print(f"  {col}: {example[col]} (type: {type(example[col])})")
        else:
            print(f"  {col} not found in example.")

In [ ]:
def tokenize_function(examples):
    """Tokenize text data for causal language modeling."""
    # Concatenate multiple columns for each example in the batch
    text = []
    for q, opt, ans in zip(examples["prompt_question"], examples["prompt_option"], examples["answer"]):
        q_str = str(q) if q is not None else ""
        opt_str = str(opt) if opt is not None else ""
        ans_str = str(ans) if ans is not None else ""
        text.append(q_str + " " + opt_str + " " + ans_str)

    # Tokenize the text
    outputs = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,  # Don't pad here; data collator will handle it
        return_overflowing_tokens=False,
    )
    return outputs

# Apply tokenization to streaming dataset
print("Tokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["prompt_question", "prompt_option", "answer", "data_idx", "question_idx", "country"] # Remove original columns
)

print(f"Preparing dataset for {MAX_STEPS} training steps...")

# Tokenized dataset

In [ ]:
tokenized_dataset

# DATA COLLATOR

In [ ]:
# Data collator for causal language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM, not masked LM
)

# TRAINING ARGUMENTS

In [ ]:
print("\nSetting up training arguments...")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS,  # Limit for demo
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,  # Keep only 2 checkpoints to save disk space
    fp16=False,  # Don't use fp16 with T4
    bf16=True,  # Use bf16 for better stability
    optim="paged_adamw_8bit",  # Memory-efficient optimizer
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="none",  # Disable W&B/tensorboard; set to "tensorboard" if needed
    gradient_checkpointing=True,  # Enable gradient checkpointing to save memory
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
)

# TRAINER

In [ ]:
print("\nInitializing Trainer...")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['test'], # Select the 'train' split
    data_collator=data_collator,
)

# Fine-tuning Llama-3.2-3B on kellycyy/CulturalBench dataset using QLoRA

In [ ]:
print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80)
print(f"Training on T4 GPU with {PER_DEVICE_TRAIN_BATCH_SIZE} batch size")
print(f"Gradient accumulation steps: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch size: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Max training steps: {MAX_STEPS}")
print(f"Sequence length: {MAX_SEQ_LENGTH}")
print("="*80 + "\n")

# Check GPU memory before training
if torch.cuda.is_available():
    print(f"GPU Memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU Memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print()

# Train the model
trainer.train()

# SAVE MODEL

In [ ]:
print("\n" + "="*80)
print("TRAINING COMPLETED - SAVING MODEL")
print("="*80)

# Save the fine-tuned model
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model saved to: {OUTPUT_DIR}")

# Save merged model (optional - combines base model with LoRA adapters)
print("\nTo use the model, load it with:")
print(f"  from peft import PeftModel")
print(f"  base_model = AutoModelForCausalLM.from_pretrained('{MODEL_NAME}')")
print(f"  model = PeftModel.from_pretrained(base_model, '{OUTPUT_DIR}')")

### Preparing .zip of finetuned model and uploading to Google drive

In [ ]:
from google.colab import files
import os

output_zip_file = "/content/llama-3.2-3b-finetuned-CulturalBench.zip"
source_dir = "./llama-3.2-3b-finetuned-CulturalBench"

# Zip the entire directory
print(f"Zipping directory: {source_dir} to {output_zip_file}")
!zip -r "$output_zip_file" "$source_dir"

# Check if the zip file was created
if os.path.exists(output_zip_file):
    print(f"Initiating download of {output_zip_file}")
    files.download(output_zip_file)
else:
    print(f"Error: Zip file not created at {output_zip_file}")

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
print("Mounting Google Drive...")
try:
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully.")
except Exception as e:
    print(f"❌ Error mounting Google Drive: {e}")

# Define the source file path and destination directory in Drive
source_file_path = "/content/llama-3.2-3b-finetuned-CulturalBench.zip"
destination_dir_in_drive = "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels" # You can change this path

# Ensure the destination directory exists in Drive
if os.path.exists("/content/drive"):
    destination_path = os.path.join(destination_dir_in_drive, os.path.basename(source_file_path))
    os.makedirs(destination_dir_in_drive, exist_ok=True)

    # Check if the source file exists before attempting to copy
    if os.path.exists(source_file_path):
        print(f"\nCopying '{source_file_path}' to '{destination_path}'...")
        try:
            # Use shell command for copying
            !cp "$source_file_path" "$destination_path"
            print("✅ File copied to Google Drive successfully!")
        except Exception as e:
            print(f"❌ Error copying file to Google Drive: {e}")
    else:
        print(f"❌ Error: Source file '{source_file_path}' not found.")
else:
    print("❌ Google Drive not mounted. Cannot copy file.")

Unzip the uploaded finetuned zipped model file from Google Drive

In [ ]:
from google.colab import drive
import os
import shutil

# Mount Google Drive if not already mounted
if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    try:
        drive.mount('/content/drive')
        print("✅ Google Drive mounted successfully.")
    except Exception as e:
        print(f"❌ Error mounting Google Drive: {e}")
else:
    print("Google Drive already mounted.")

# Define the source file path in Drive and destination directory in /content
source_file_path_in_drive = "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/llama-3.2-3b-finetuned-CulturalBench.zip"
destination_dir_in_content = "/content/"
destination_file_path_in_content = os.path.join(destination_dir_in_content, os.path.basename(source_file_path_in_drive))

# Ensure the destination directory exists in /content (it should)
os.makedirs(destination_dir_in_content, exist_ok=True)

# Check if the source file exists in Drive before attempting to copy
if os.path.exists(source_file_path_in_drive):
    print(f"\nCopying '{source_file_path_in_drive}' to '{destination_file_path_in_content}'...")
    try:
        shutil.copy2(source_file_path_in_drive, destination_file_path_in_content)
        print("✅ File copied from Google Drive to /content successfully!")

        # Now, unzip the file in /content
        print(f"\nUnzipping '{destination_file_path_in_content}' to '{destination_dir_in_content}'...")
        try:
            !unzip -q "$destination_file_path_in_content" -d "$destination_dir_in_content"
            print("✅ File unzipped successfully!")
        except Exception as e:
            print(f"❌ Error unzipping file: {e}")

    except Exception as e:
        print(f"❌ Error copying file from Google Drive: {e}")
else:
    print(f"❌ Error: Source file '{source_file_path_in_drive}' not found in Google Drive.")

nDNA analysis

In [ ]:
%%capture
!pip uninstall -y transformers datasets accelerate peft torch matplotlib numpy scipy plotly scikit-learn bitsandbytes
!pip install -q -U \
    numpy \
    scipy \
    scikit-learn \
    torch \
    transformers \
    datasets \
    accelerate \
    peft \
    matplotlib \
    plotly \
    bitsandbytes

# ============================================================================
# CELL 2: Imports
# ============================================================================
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

import math
import random
from tqdm import tqdm
from functools import partial
import warnings
warnings.filterwarnings('ignore')

In [ ]:


# Model paths
BASE_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
FINETUNED_MODEL_PATH = "./llama-3.2-3b-finetuned-CulturalBench"  # Your local path

# Device configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

# Dataset configuration
MAX_SAMPLES_SQUAD = 200          # SQuAD v2 samples
MAX_SAMPLES_CULTURAL = 150       # CulturalBench samples per category
BATCH_SIZE = 2                   # Small batch for T4
MAX_SEQ_LEN = 256               # Moderate sequence length

# nDNA Analysis parameters
TOKENS_PER_EXAMPLE = 32         # For belief vectors
TAU = 1.0                       # Temperature for softmax
FR_NORM = True                  # Fisher-Rao normalization
EPS_DIST = 1e-12               # Numerical stability
EPS_CURV = 1e-12               # Curvature epsilon

# Mixed precision
AMP_DTYPE = (torch.bfloat16 if (DEVICE=="cuda" and torch.cuda.is_bf16_supported())
             else torch.float16)

print(f"✅ Device: {DEVICE}")
print(f"✅ AMP dtype: {AMP_DTYPE}")
# print(f"✅ Base model: {BASE_MODEL_NAME}")
print(f"✅ Fine-tuned model: {FINETUNED_MODEL_PATH}")
print(f"✅ Fisher-Rao norm: {FR_NORM}")

# ============================================================================
# CELL 4: Set Random Seeds
# ============================================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("✅ Random seeds set for reproducibility")

# ============================================================================
# CELL 5: Load Fine-Tuned Model
# ============================================================================
print("\n" + "="*80)
print("LOADING FINE-TUNED MODEL")
print("="*80)

# print(f"Loading tokenizer from {BASE_MODEL_NAME}...")
# tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
# if tokenizer.pad_token_id is None:
#     tokenizer.pad_token = tokenizer.eos_token
#     tokenizer.pad_token_id = tokenizer.eos_token_id
# tokenizer.padding_side = "right"

print(f"Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

print(f"Loading LoRA adapters from {FINETUNED_MODEL_PATH}...")
model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL_PATH)
model.eval()

# # Get model components (LLaMA architecture)
# llama_model = model.base_model.model.model  # Access the base LlamaModel
# blocks = llama_model.layers                  # Decoder layers
# final_norm = llama_model.norm               # Final RMSNorm
# lm_head = model.base_model.model.lm_head   # Language modeling head

# num_layers = len(blocks)
# vocab_size = model.config.vocab_size

# print(f"✅ Model loaded successfully!")
# print(f"   Layers: {num_layers}")
# print(f"   Vocabulary size: {vocab_size}")
# print(f"   Model dtype: {next(model.parameters()).dtype}")

# ============================================================================
# CELL 6: Load Datasets
# ============================================================================
print("\n" + "="*80)
print("LOADING DATASETS")
print("="*80)

# ===== SQuAD v2 Dataset =====
print("\n📚 Loading SQuAD v2...")
squad_raw = load_dataset("squad_v2", split="train")

def format_squad(example):
    """Format SQuAD example as prompt"""
    question = example["question"].strip()
    context = example["context"].strip()
    return {"text": f"Question: {question}\nContext: {context}\nAnswer:"}

squad_dataset = squad_raw.map(format_squad, remove_columns=squad_raw.column_names)
squad_dataset = squad_dataset.filter(lambda ex: len(ex["text"]) > 50)
if MAX_SAMPLES_SQUAD:
    squad_dataset = squad_dataset.select(range(min(MAX_SAMPLES_SQUAD, len(squad_dataset))))

print(f"✅ SQuAD v2: {len(squad_dataset)} samples")
print(f"   Sample: {squad_dataset[0]['text'][:150]}...")

# ===== CulturalBench Dataset =====
print("\n🌍 Loading CulturalBench...")
cultural_raw = load_dataset("kellycyy/CulturalBench", "CulturalBench-Hard")

def format_cultural(example):
    """Format CulturalBench example"""
    question = str(example.get("prompt_question", "")).strip()
    options = str(example.get("prompt_option", "")).strip()
    return {"text": f"{question} {options}".strip(), "country": example.get("country", "Unknown")}

cultural_dataset = cultural_raw["test"].map(
    format_cultural,
    remove_columns=[col for col in cultural_raw["test"].column_names if col not in ["country"]]
)
cultural_dataset = cultural_dataset.filter(lambda ex: len(ex["text"]) > 20)

# Group by country for analysis
countries = list(set(cultural_dataset["country"]))
cultural_by_country = {
    country: cultural_dataset.filter(lambda ex: ex["country"] == country)
    for country in countries[:5]  # Top 5 countries for visualization
}

print(f"✅ CulturalBench: {len(cultural_dataset)} samples")
print(f"   Countries: {len(countries)}")
print(f"   Top countries: {list(cultural_by_country.keys())}")
print(f"   Sample: {cultural_dataset[0]['text'][:150]}...")

# ============================================================================
# CELL 7: Data Collation Functions
# ============================================================================
print("\n" + "="*80)
print("SETTING UP DATA COLLATORS")
print("="*80)

def causal_collate_standard(batch):
    """Standard collation for thermodynamic length (full sequence)"""
    texts = [ex["text"] for ex in batch]
    tok = tokenizer(
        texts,
        padding="longest",
        truncation=True,
        max_length=MAX_SEQ_LEN,
        return_tensors="pt"
    )

    input_ids = tok["input_ids"]
    attention_mask = tok["attention_mask"]

    # Shift for next-token prediction
    x = input_ids[:, :-1].contiguous()
    y = input_ids[:, 1:].contiguous()
    attn = attention_mask[:, :-1].contiguous()

    # Mask padding in labels
    y = y.masked_fill(attention_mask[:, 1:] == 0, -100)

    return {
        "input_ids": x,
        "attention_mask": attn,
        "labels": y
    }

def causal_collate_belief(batch, keep_last_k=TOKENS_PER_EXAMPLE):
    """Collation for belief vectors (keep last K tokens)"""
    texts = [ex["text"] for ex in batch]
    tok = tokenizer(
        texts,
        padding="longest",
        truncation=True,
        max_length=MAX_SEQ_LEN,
        return_tensors="pt"
    )

    input_ids = tok["input_ids"]
    attention_mask = tok["attention_mask"]

    x = input_ids[:, :-1].contiguous()
    y = input_ids[:, 1:].contiguous()
    attn = attention_mask[:, :-1].contiguous()
    y = y.masked_fill(attention_mask[:, 1:] == 0, -100)

    # Select mask: keep only last K supervised positions
    B, S = x.shape
    sel_mask = torch.zeros_like(y, dtype=torch.bool)
    for b in range(B):
        valid = (y[b] != -100).nonzero(as_tuple=False).squeeze(-1)
        if valid.numel() > 0:
            take = valid[-min(keep_last_k, valid.numel()):]
            sel_mask[b, take] = True

    return {
        "input_ids": x,
        "attention_mask": attn,
        "labels": y,
        "select_mask": sel_mask
    }

print("✅ Data collators configured")

# ============================================================================
# CELL 8: Geometry Helper Functions
# ============================================================================
print("\n" + "="*80)
print("DEFINING GEOMETRY HELPERS")
print("="*80)

def sqrt_embed(q: torch.Tensor, eps: float = EPS_DIST) -> torch.Tensor:
    """
    Map probability distribution q to unit sphere via sqrt embedding
    Returns: u = sqrt(q) / ||sqrt(q)||
    """
    q = torch.clamp(q, min=eps)
    q = q / q.sum()
    u = torch.sqrt(q)
    u = u / (torch.norm(u, p=2) + 1e-30)
    return u

def project_tangent(u: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """
    Project vector v onto tangent space at u on the sphere
    Returns: v - <u,v>u
    """
    return v - torch.dot(u, v) * u

def fisher_rao_distance(logp1: torch.Tensor, logp2: torch.Tensor) -> torch.Tensor:
    """
    Compute Fisher-Rao (Bhattacharyya) distance between two log-probability distributions
    d_FR(p,q) = 2 * arccos(BC(p,q)) where BC = sum(sqrt(p*q))

    Args:
        logp1, logp2: Log-probabilities (..., V)
    Returns:
        Fisher-Rao distance (...,)
    """
    # Compute log of Bhattacharyya coefficient: log(BC) = log(sum(sqrt(p*q)))
    # = logsumexp(0.5*(log(p) + log(q)))
    s = 0.5 * (logp1 + logp2)
    log_bc = torch.logsumexp(s, dim=-1)
    bc = torch.exp(log_bc).clamp(0.0, 1.0)
    return 2.0 * torch.acos(bc)

print("✅ Geometry helpers defined")

# ============================================================================
# CELL 9: METRIC 1 - Spectral Curvature
# ============================================================================
print("\n" + "="*80)
print("METRIC 1: SPECTRAL CURVATURE")
print("="*80)

@torch.no_grad()
def compute_spectral_curvature(text: str, model_name: str = "fine-tuned"):
    """
    Compute spectral curvature across layers using logit lens

    Returns:
        curvatures: array of κ^(simp) for interior layers
        speeds: array of ||Δu|| for each layer transition
    """
    # Tokenize
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=True).to(DEVICE)
    if enc.input_ids.shape[1] == 0:
        raise ValueError("Empty input")

    input_ids = enc.input_ids
    attention_mask = enc.attention_mask

    # Forward pass to get all hidden states
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states  # Tuple of (num_layers+1) tensors

    # Get last position (next-token prediction)
    t = input_ids.shape[1] - 1

    # Extract distributions at each layer via logit lens
    q_list = []
    for h in hidden_states:
        vec = h[0, t, :]
        vec_norm = final_norm(vec)
        logits = lm_head(vec_norm)
        q = torch.softmax(logits.float() / TAU, dim=-1)
        q_list.append(q)

    # Map to sphere
    u_list = [sqrt_embed(q) for q in q_list]

    # Compute curvature using finite differences
    m = len(u_list)
    if m < 3:
        return np.array([]), np.array([])

    # First differences and speeds
    delta_u = []
    speeds = []
    for ell in range(m - 1):
        u_curr = u_list[ell]
        v = u_list[ell + 1] - u_curr
        du = project_tangent(u_curr, v)
        delta_u.append(du)
        speeds.append(torch.norm(du, p=2).item())

    # Second differences and curvatures (interior points)
    curvatures = []
    for ell in range(1, m - 1):
        u_curr = u_list[ell]
        # Second difference: Δ²u_ell = u_{ell+1} - 2*u_ell + u_{ell-1}
        v2 = u_list[ell + 1] - 2 * u_list[ell] + u_list[ell - 1]
        d2u = project_tangent(u_curr, v2)

        # Curvature: κ = ||Δ²u|| / ||Δu||³
        num = torch.norm(d2u, p=2)
        s = torch.norm(delta_u[ell], p=2)
        denom = (s * s + EPS_CURV) ** 1.5
        kappa = (num / denom).item()
        curvatures.append(kappa)

    return np.array(curvatures), np.array(speeds)

# Test on sample prompts
print("\n🧪 Testing Spectral Curvature...")
test_prompts = [
    ("SQuAD Sample", squad_dataset[0]["text"]),
    ("Cultural Sample", cultural_dataset[0]["text"]),
]

curvature_results = {}
for name, text in test_prompts:
    print(f"\nProcessing: {name}")
    print(f"Text: {text[:100]}...")
    curvs, speeds = compute_spectral_curvature(text)
    curvature_results[name] = {"curvatures": curvs, "speeds": speeds}
    print(f"✅ Computed {len(curvs)} curvature values")
    if len(curvs) > 0:
        print(f"   Mean curvature: {curvs.mean():.6e}")
        print(f"   Max curvature: {curvs.max():.6e}")

# ============================================================================
# CELL 10: METRIC 2 - Thermodynamic Length (Parameter-based)
# ============================================================================
print("\n" + "="*80)
print("METRIC 2: THERMODYNAMIC LENGTH (PARAMETER-BASED)")
print("="*80)

def compute_parameter_thermodynamic_length(dataset, dataset_name: str):
    """
    Compute per-layer effort via observed Fisher (gradient norms)
    """
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=causal_collate_standard
    )

    layer_grad_sums = torch.zeros(num_layers, device=DEVICE)
    num_batches = 0

    print(f"\n🔥 Computing parameter-based thermodynamic length for {dataset_name}...")

    for batch in tqdm(loader, desc=f"[{dataset_name}]"):
        num_batches += 1

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        B, S = input_ids.shape

        # Get initial embeddings
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
            # LLaMA uses embed_tokens
            h = llama_model.embed_tokens(input_ids)

        per_layer_losses = []

        # Forward through each layer
        for ell in range(num_layers):
            h_prev = h.detach()  # Isolate gradients

            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
                # Forward one layer
                layer_output = blocks[ell](
                    h_prev,
                    attention_mask=attention_mask,
                )
                h_ell = layer_output[0]

                # Logit lens
                logits_ell = lm_head(final_norm(h_ell))

                # Cross-entropy loss
                loss_ell = F.cross_entropy(
                    logits_ell.view(-1, vocab_size),
                    labels.view(-1),
                    ignore_index=-100,
                    reduction="mean"
                )

            per_layer_losses.append(loss_ell)
            h = h_ell

        # Single backward for all layers
        model.zero_grad(set_to_none=True)
        total_loss = torch.stack(per_layer_losses).sum()
        total_loss.backward()

        # Accumulate gradient norms
        with torch.no_grad():
            for ell in range(num_layers):
                g2 = torch.tensor(0.0, device=DEVICE)
                # Get parameters for this layer
                for p in blocks[ell].parameters():
                    if p.grad is not None:
                        g2 += (p.grad.detach() ** 2).sum()
                layer_grad_sums[ell] += g2

    # Average over batches
    mean_grad_norms = (layer_grad_sums / max(1, num_batches)).cpu().numpy()

    print(f"✅ Computed parameter-based thermodynamic length")
    print(f"   Batches processed: {num_batches}")
    print(f"   Mean gradient norm (first layer): {mean_grad_norms[0]:.6e}")
    print(f"   Mean gradient norm (last layer): {mean_grad_norms[-1]:.6e}")

    return mean_grad_norms

# Compute for SQuAD
squad_param_thermo = compute_parameter_thermodynamic_length(squad_dataset, "SQuAD v2")

# ============================================================================
# CELL 11: METRIC 2 - Thermodynamic Length (Prediction-based, Fisher-Rao)
# ============================================================================
print("\n" + "="*80)
print("METRIC 2: THERMODYNAMIC LENGTH (PREDICTION-BASED, FISHER-RAO)")
print("="*80)

@torch.no_grad()
def compute_prediction_thermodynamic_length(dataset, dataset_name: str):
    """
    Compute Fisher-Rao distance between consecutive layer predictions
    """
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=causal_collate_standard
    )

    num_steps = num_layers - 1
    fr_step_sums = torch.zeros(num_steps, device=DEVICE)
    fr_step_counts = torch.zeros(num_steps, device=DEVICE)

    print(f"\n🔥 Computing prediction-based thermodynamic length for {dataset_name}...")

    for batch in tqdm(loader, desc=f"[{dataset_name}]"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        B, S = input_ids.shape

        # Get initial embeddings
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
            h = llama_model.embed_tokens(input_ids)

        logp_prev = None

        for ell in range(num_layers):
            # Forward one layer
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
                layer_output = blocks[ell](h, attention_mask=attention_mask)
                h = layer_output[0]

                # Logit lens
                logits = lm_head(final_norm(h))
                logp = F.log_softmax(logits.float(), dim=-1)

            if logp_prev is not None:
                # Valid positions (not padding)
                valid = (labels != -100)

                # Fisher-Rao distance
                fr_dists = fisher_rao_distance(logp_prev, logp)  # (B, S)

                # Accumulate
                step_idx = ell - 1
                fr_step_sums[step_idx] += fr_dists.masked_fill(~valid, 0.0).sum()
                fr_step_counts[step_idx] += valid.sum()

            logp_prev = logp

    # Average
    fr_step_means = (fr_step_sums / fr_step_counts.clamp_min(1)).cpu().numpy()

    print(f"✅ Computed prediction-based thermodynamic length")
    print(f"   Mean FR distance (first step): {fr_step_means[0]:.6e}")
    print(f"   Mean FR distance (last step): {fr_step_means[-1]:.6e}")

    return fr_step_means

# Compute for SQuAD
squad_pred_thermo = compute_prediction_thermodynamic_length(squad_dataset, "SQuAD v2")

# ============================================================================
# CELL 12: METRIC 3 - Belief Vector Fields
# ============================================================================
print("\n" + "="*80)
print("METRIC 3: BELIEF VECTOR FIELDS")
print("="*80)

@torch.no_grad()
def compute_belief_vectors(dataset, dataset_name: str):
    """
    Compute belief vector field magnitudes across layers
    """
    loader = DataLoader(
        dataset,
        batch_size=1,  # Batch size 1 for belief vectors
        shuffle=False,
        collate_fn=causal_collate_belief
    )

    # Accumulators
    v_sum = [torch.zeros(vocab_size, device=DEVICE, dtype=torch.float32) for _ in range(num_layers)]
    u_sum = [torch.zeros(vocab_size, device=DEVICE, dtype=torch.float32) for _ in range(num_layers)]
    cnt = [0 for _ in range(num_layers)]

    print(f"\n🔥 Computing belief vectors for {dataset_name}...")

    for batch in tqdm(loader, desc=f"[{dataset_name}]"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        keep_mask = batch["select_mask"].to(DEVICE)
        B, S = input_ids.shape

        # Get all hidden states
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
                output_hidden_states=True
            )
            hidden_states = outputs.hidden_states

        # For each layer
        for ell in range(num_layers):
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
                h_ell = hidden_states[ell + 1]  # +1 because index 0 is embeddings
                z = lm_head(final_norm(h_ell))

            # Keep only selected positions
            keep = keep_mask.view(-1)
            if not keep.any():
                continue

            z_sel = z.view(-1, vocab_size)[keep]
            y_sel = labels.view(-1)[keep]

            # Compute q, u in float32
            z32 = z_sel.float() / TAU
            q = F.softmax(z32, dim=-1)
            q = torch.clamp(q, min=1e-12)
            u = torch.sqrt(q)

            # Gradient g = (1/τ)(e_y - q)
            g = -q / TAU
            add = torch.full((y_sel.shape[0], 1), 1.0/TAU, device=DEVICE, dtype=q.dtype)
            g.scatter_add_(dim=-1, index=y_sel.unsqueeze(-1), src=add)

            # Tangent vector t = (1/(2τ)) * [Diag(u)g - u(q^T g)]
            ug = u * g
            s = (q * g).sum(dim=-1, keepdim=True)
            t = (ug - s * u) / (2.0 * TAU)

            # Accumulate
            v_sum[ell] += t.sum(dim=0).to(v_sum[ell].dtype)
            u_sum[ell] += u.sum(dim=0).to(u_sum[ell].dtype)
            cnt[ell] += t.size(0)

    # Finalize: average, rebase, measure norm
    norms = []
    for ell in range(num_layers):
        if cnt[ell] == 0:
            norms.append(0.0)
            continue

        v_avg = v_sum[ell] / cnt[ell]
        u_avg = u_sum[ell] / cnt[ell]

        # Normalize u_avg
        u_norm = torch.linalg.norm(u_avg).clamp_min(1e-12)
        u_bar = u_avg / u_norm

        # Project v into tangent space
        radial = torch.dot(u_bar, v_avg)
        v_tan = v_avg - radial * u_bar

        # Measure norm
        n = torch.linalg.norm(v_tan).item()
        norms.append(2.0 * n if FR_NORM else n)

    print(f"✅ Computed belief vectors")
    print(f"   Positions processed per layer: {cnt[0]} (first), {cnt[-1]} (last)")

    return np.array(norms)

# Compute for SQuAD
squad_belief = compute_belief_vectors(squad_dataset, "SQuAD v2")

# Compute for top 3 cultural countries
cultural_belief_results = {}
for country in list(cultural_by_country.keys())[:3]:
    ds = cultural_by_country[country]
    if len(ds) > MAX_SAMPLES_CULTURAL:
        ds = ds.select(range(MAX_SAMPLES_CULTURAL))
    belief_norms = compute_belief_vectors(ds, f"Cultural-{country}")
    cultural_belief_results[country] = belief_norms

# ============================================================================
# CELL 13: Comprehensive Visualization - Part 1 (Spectral Curvature)
# ============================================================================
print("\n" + "="*80)
print("GENERATING VISUALIZATIONS - SPECTRAL CURVATURE")
print("="*80)

# Compute spectral curvature for multiple samples
print("\n🎨 Computing spectral curvature for multiple samples...")

squad_curvatures = []
for i in tqdm(range(min(20, len(squad_dataset))), desc="SQuAD curvature"):
    curvs, _ = compute_spectral_curvature(squad_dataset[i]["text"])
    if len(curvs) > 0:
        squad_curvatures.append(curvs)

cultural_curvatures = []
for i in tqdm(range(min(20, len(cultural_dataset))), desc="Cultural curvature"):
    curvs, _ = compute_spectral_curvature(cultural_dataset[i]["text"])
    if len(curvs) > 0:
        cultural_curvatures.append(curvs)

# Average curvatures
squad_curv_mean = np.mean(squad_curvatures, axis=0) if squad_curvatures else np.array([])
cultural_curv_mean = np.mean(cultural_curvatures, axis=0) if cultural_curvatures else np.array([])

# Plot
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
if len(squad_curv_mean) > 0:
    layers = np.arange(1, len(squad_curv_mean) + 1)
    plt.plot(layers, squad_curv_mean, marker='o', color='blue', label='SQuAD v2')
plt.title('Spectral Curvature - SQuAD v2\n(Fine-tuned Model)', fontsize=14, fontweight='bold')
plt.xlabel('Interior Layer Index', fontsize=12)
plt.ylabel('κ^(simp) (Curvature)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
if len(cultural_curv_mean) > 0:
    layers = np.arange(1, len(cultural_curv_mean) + 1)
    plt.plot(layers, cultural_curv_mean, marker='s', color='green', label='CulturalBench')
plt.title('Spectral Curvature - CulturalBench\n(Fine-tuned Model)', fontsize=14, fontweight='bold')
plt.xlabel('Interior Layer Index', fontsize=12)
plt.ylabel('κ^(simp) (Curvature)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

print("✅ Spectral curvature plots generated")

# ============================================================================
# CELL 14: Comprehensive Visualization - Part 2 (Thermodynamic Length)
# ============================================================================
print("\n" + "="*80)
print("GENERATING VISUALIZATIONS - THERMODYNAMIC LENGTH")
print("="*80)

# Create comprehensive thermodynamic length plot
fig = plt.figure(figsize=(18, 12))
gs = GridSpec(3, 2, figure=fig, hspace=0.3, wspace=0.3)

# Plot 1: Parameter-based (SQuAD)
ax1 = fig.add_subplot(gs[0, 0])
layers_full = np.arange(1, num_layers + 1)
ax1.plot(layers_full, squad_param_thermo, marker='o', color='red', linewidth=2)
ax1.set_title('Parameter-Based Thermodynamic Length\nSQuAD v2 (Internal Effort)',
              fontsize=13, fontweight='bold')
ax1.set_xlabel('Layer ℓ', fontsize=11)
ax1.set_ylabel('Mean Squared Grad Norm (a.u.)', fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# Plot 2: Prediction-based (SQuAD)
ax2 = fig.add_subplot(gs[0, 1])
steps = np.arange(1, len(squad_pred_thermo) + 1)
ax2.plot(steps, squad_pred_thermo, marker='^', color='blue', linewidth=2)
ax2.set_title('Prediction-Based Thermodynamic Length\nSQuAD v2 (Fisher-Rao Distance)',
              fontsize=13, fontweight='bold')
ax2.set_xlabel('Layer Transition (ℓ → ℓ+1)', fontsize=11)
ax2.set_ylabel('Mean FR Distance (radians)', fontsize=11)
ax2.grid(True, alpha=0.3)

# Plot 3: Semantic Efficiency (Combined metric)
ax3 = fig.add_subplot(gs[1, :])
# Normalize both metrics
param_norm = (squad_param_thermo - squad_param_thermo.min()) / (squad_param_thermo.max() - squad_param_thermo.min() + 1e-9)
pred_norm_full = np.zeros(num_layers)
pred_norm_full[:-1] = (squad_pred_thermo - squad_pred_thermo.min()) / (squad_pred_thermo.max() - squad_pred_thermo.min() + 1e-9)
pred_norm_full[-1] = pred_norm_full[-2]  # Extend last value

# Semantic efficiency: log(1 + belief_change / parameter_effort)
semantic_eff = np.log(1 + pred_norm_full / (param_norm + 1e-9))
ax3.plot(layers_full, semantic_eff, marker='D', color='green', linewidth=2.5, markersize=8)
ax3.set_title('Semantic Efficiency (Unified Metric)\nlog(1 + Norm(Δp_ℓ) / Norm(E_ℓ))',
              fontsize=14, fontweight='bold')
ax3.set_xlabel('Layer ℓ', fontsize=12)
ax3.set_ylabel('Semantic Efficiency Score', fontsize=12)
ax3.grid(True, alpha=0.3)
ax3.fill_between(layers_full, 0, semantic_eff, alpha=0.2, color='green')

# Plot 4: Layer-wise comparison
ax4 = fig.add_subplot(gs[2, :])
ax4_twin = ax4.twinx()

# Parameter effort (left axis, log scale)
line1 = ax4.plot(layers_full, squad_param_thermo, marker='o', color='red',
                 linewidth=2, label='Parameter Effort (E_ℓ)')
ax4.set_xlabel('Layer ℓ', fontsize=12)
ax4.set_ylabel('Parameter Effort (log scale)', fontsize=11, color='red')
ax4.tick_params(axis='y', labelcolor='red')
ax4.set_yscale('log')
ax4.grid(True, alpha=0.2)

# Prediction change (right axis, linear scale)
pred_full = np.zeros(num_layers)
pred_full[:-1] = squad_pred_thermo
pred_full[-1] = squad_pred_thermo[-1]
line2 = ax4_twin.plot(layers_full, pred_full, marker='^', color='blue',
                      linewidth=2, label='Belief Change (Δp_ℓ)')
ax4_twin.set_ylabel('Belief Change (radians)', fontsize=11, color='blue')
ax4_twin.tick_params(axis='y', labelcolor='blue')

# Combined legend
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax4.legend(lines, labels, loc='upper right', fontsize=10)
ax4.set_title('Layer-wise Comparison: Parameter Effort vs. Belief Change',
              fontsize=13, fontweight='bold')

plt.suptitle(f'Thermodynamic Length Analysis - Fine-tuned Llama on SQuAD v2\n'
             f'{num_layers} Layers | {len(squad_dataset)} Samples',
             fontsize=16, fontweight='bold', y=0.995)

plt.show()

print("✅ Thermodynamic length plots generated")

# ============================================================================
# CELL 15: Comprehensive Visualization - Part 3 (Belief Vectors)
# ============================================================================
print("\n" + "="*80)
print("GENERATING VISUALIZATIONS - BELIEF VECTORS")
print("="*80)

# Create belief vector plot
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Belief Vector Field Magnitudes Across Layers\nFine-tuned Llama Model',
             fontsize=16, fontweight='bold')

# Plot 1: SQuAD
ax = axes[0, 0]
layers = np.arange(1, num_layers + 1)
ax.plot(layers, squad_belief, marker='o', color='blue', linewidth=2, markersize=6)
ax.set_title('SQuAD v2 Dataset', fontsize=13, fontweight='bold')
ax.set_xlabel('Layer Index', fontsize=11)
ax.set_ylabel('||v_ℓ|| (Fisher-Rao norm)' if FR_NORM else '||v_ℓ|| (Euclidean norm)', fontsize=11)
ax.grid(True, alpha=0.3)
ax.fill_between(layers, 0, squad_belief, alpha=0.2, color='blue')

# Plot 2: Cultural samples (overlay multiple countries)
ax = axes[0, 1]
colors_cultural = ['green', 'orange', 'purple']
for i, (country, norms) in enumerate(cultural_belief_results.items()):
    ax.plot(layers, norms, marker='s', color=colors_cultural[i],
            linewidth=2, label=country, markersize=6)
ax.set_title('CulturalBench (by Country)', fontsize=13, fontweight='bold')
ax.set_xlabel('Layer Index', fontsize=11)
ax.set_ylabel('||v_ℓ|| (Fisher-Rao norm)' if FR_NORM else '||v_ℓ|| (Euclidean norm)', fontsize=11)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)

# Plot 3: Comparison (SQuAD vs Cultural average)
ax = axes[1, 0]
cultural_avg = np.mean(list(cultural_belief_results.values()), axis=0)
ax.plot(layers, squad_belief, marker='o', color='blue', linewidth=2,
        label='SQuAD v2', markersize=6)
ax.plot(layers, cultural_avg, marker='s', color='green', linewidth=2,
        label='CulturalBench (avg)', markersize=6)
ax.set_title('Dataset Comparison', fontsize=13, fontweight='bold')
ax.set_xlabel('Layer Index', fontsize=11)
ax.set_ylabel('||v_ℓ|| (Fisher-Rao norm)' if FR_NORM else '||v_ℓ|| (Euclidean norm)', fontsize=11)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)

# Plot 4: Normalized comparison (show relative patterns)
ax = axes[1, 1]
squad_norm = (squad_belief - squad_belief.min()) / (squad_belief.max() - squad_belief.min() + 1e-9)
cultural_norm = (cultural_avg - cultural_avg.min()) / (cultural_avg.max() - cultural_avg.min() + 1e-9)
ax.plot(layers, squad_norm, marker='o', color='blue', linewidth=2,
        label='SQuAD v2 (normalized)', markersize=6)
ax.plot(layers, cultural_norm, marker='s', color='green', linewidth=2,
        label='CulturalBench (normalized)', markersize=6)
ax.set_title('Normalized Comparison (Relative Patterns)', fontsize=13, fontweight='bold')
ax.set_xlabel('Layer Index', fontsize=11)
ax.set_ylabel('Normalized Magnitude', fontsize=11)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
ax.set_ylim([-0.05, 1.05])

plt.tight_layout()
plt.show()

print("✅ Belief vector plots generated")

# ============================================================================
# CELL 16: Interactive Plotly Visualization
# ============================================================================
print("\n" + "="*80)
print("GENERATING INTERACTIVE PLOTLY VISUALIZATIONS")
print("="*80)

# Create interactive multi-panel plot
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Spectral Curvature - SQuAD v2',
        'Spectral Curvature - CulturalBench',
        'Thermodynamic Length (Parameter)',
        'Thermodynamic Length (Prediction)',
        'Belief Vectors - SQuAD v2',
        'Belief Vectors - CulturalBench'
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

# Spectral Curvature - SQuAD
if len(squad_curv_mean) > 0:
    fig.add_trace(
        go.Scatter(x=np.arange(1, len(squad_curv_mean)+1), y=squad_curv_mean,
                   mode='lines+markers', name='SQuAD Curvature',
                   line=dict(color='blue', width=2), marker=dict(size=8)),
        row=1, col=1
    )

# Spectral Curvature - Cultural
if len(cultural_curv_mean) > 0:
    fig.add_trace(
        go.Scatter(x=np.arange(1, len(cultural_curv_mean)+1), y=cultural_curv_mean,
                   mode='lines+markers', name='Cultural Curvature',
                   line=dict(color='green', width=2), marker=dict(size=8)),
        row=1, col=2
    )

# Thermodynamic Length - Parameter
fig.add_trace(
    go.Scatter(x=np.arange(1, num_layers+1), y=squad_param_thermo,
               mode='lines+markers', name='Param Effort',
               line=dict(color='red', width=2), marker=dict(size=7)),
    row=2, col=1
)

# Thermodynamic Length - Prediction
fig.add_trace(
    go.Scatter(x=np.arange(1, len(squad_pred_thermo)+1), y=squad_pred_thermo,
               mode='lines+markers', name='Belief Change',
               line=dict(color='blue', width=2), marker=dict(size=7)),
    row=2, col=2
)

# Belief Vectors - SQuAD
fig.add_trace(
    go.Scatter(x=np.arange(1, num_layers+1), y=squad_belief,
               mode='lines+markers', name='SQuAD Belief',
               line=dict(color='blue', width=2), marker=dict(size=7)),
    row=3, col=1
)

# Belief Vectors - Cultural (average)
cultural_avg = np.mean(list(cultural_belief_results.values()), axis=0)
fig.add_trace(
    go.Scatter(x=np.arange(1, num_layers+1), y=cultural_avg,
               mode='lines+markers', name='Cultural Belief',
               line=dict(color='green', width=2), marker=dict(size=7)),
    row=3, col=2
)

# Update layout
fig.update_xaxes(title_text="Layer Index", row=1, col=1)
fig.update_xaxes(title_text="Layer Index", row=1, col=2)
fig.update_xaxes(title_text="Layer ℓ", row=2, col=1)
fig.update_xaxes(title_text="Layer Transition", row=2, col=2)
fig.update_xaxes(title_text="Layer Index", row=3, col=1)
fig.update_xaxes(title_text="Layer Index", row=3, col=2)

fig.update_yaxes(title_text="κ^(simp)", row=1, col=1)
fig.update_yaxes(title_text="κ^(simp)", row=1, col=2)
fig.update_yaxes(title_text="Grad Norm", type="log", row=2, col=1)
fig.update_yaxes(title_text="FR Distance", row=2, col=2)
fig.update_yaxes(title_text="||v_ℓ||", row=3, col=1)
fig.update_yaxes(title_text="||v_ℓ||", row=3, col=2)

fig.update_layout(
    height=1200,
    title_text="Complete nDNA Analysis - Fine-tuned Llama Model",
    showlegend=True,
    title_font_size=18
)

fig.show()

print("✅ Interactive plots generated")

# ============================================================================
# CELL 17: Summary Statistics and Report
# ============================================================================
print("\n" + "="*80)
print("FINAL SUMMARY AND STATISTICS")
print("\n📊 SPECTRAL CURVATURE")
print("-" * 60)
if len(squad_curv_mean) > 0:
    print(f"SQuAD v2:")
    print(f"  Mean curvature: {squad_curv_mean.mean():.6e}")
    print(f"  Max curvature: {squad_curv_mean.max():.6e}")
    print(f"  Min curvature: {squad_curv_mean.min():.6e}")
    print(f"  Std deviation: {squad_curv_mean.std():.6e}")

if len(cultural_curv_mean) > 0:
    print(f"\nCulturalBench:")
    print(f"  Mean curvature: {cultural_curv_mean.mean():.6e}")
    print(f"  Max curvature: {cultural_curv_mean.max():.6e}")
    print(f"  Min curvature: {cultural_curv_mean.min():.6e}")
    print(f"  Std deviation: {cultural_curv_mean.std():.6e}")

print("\n📏 THERMODYNAMIC LENGTH")
print("-" * 60)
print("Parameter-based (SQuAD v2):")
print(f"  Total effort: {squad_param_thermo.sum():.6e}")
print(f"  Mean per layer: {squad_param_thermo.mean():.6e}")
print(f"  Max layer: {squad_param_thermo.max():.6e} (Layer {squad_param_thermo.argmax()+1})")
print(f"  Min layer: {squad_param_thermo.min():.6e} (Layer {squad_param_thermo.argmin()+1})")

print("\nPrediction-based (SQuAD v2):")
print(f"  Total FR distance: {squad_pred_thermo.sum():.6e} radians")
print(f"  Mean per step: {squad_pred_thermo.mean():.6e} radians")
print(f"  Max step: {squad_pred_thermo.max():.6e} (Step {squad_pred_thermo.argmax()+1})")
print(f"  Min step: {squad_pred_thermo.min():.6e} (Step {squad_pred_thermo.argmin()+1})")

print("\n🎯 BELIEF VECTORS")
print("-" * 60)
print("SQuAD v2:")
print(f"  Mean magnitude: {squad_belief.mean():.6e}")
print(f"  Max magnitude: {squad_belief.max():.6e} (Layer {squad_belief.argmax()+1})")
print(f"  Min magnitude: {squad_belief.min():.6e} (Layer {squad_belief.argmin()+1})")
print(f"  Std deviation: {squad_belief.std():.6e}")

print("\nCulturalBench (average across countries):")
cultural_avg = np.mean(list(cultural_belief_results.values()), axis=0)
print(f"  Mean magnitude: {cultural_avg.mean():.6e}")
print(f"  Max magnitude: {cultural_avg.max():.6e} (Layer {cultural_avg.argmax()+1})")
print(f"  Min magnitude: {cultural_avg.min():.6e} (Layer {cultural_avg.argmin()+1})")
print(f"  Std deviation: {cultural_avg.std():.6e}")

print("\n" + "="*80)
print("✅ COMPLETE nDNA ANALYSIS FINISHED")
print("="*80)
print(f"\nAnalyzed Model: Fine-tuned {BASE_MODEL_NAME}")
print(f"Datasets: SQuAD v2 ({len(squad_dataset)} samples), CulturalBench ({len(cultural_dataset)} samples)")
print(f"Layers analyzed: {num_layers}")
print(f"All metrics computed successfully!")
print("\n💡 Key Observations:")
print("   1. Spectral curvature shows geometric complexity of representation space")
print("   2. Thermodynamic length reveals both parameter and prediction effort")
print("   3. Belief vectors capture semantic information flow across layers")
print("   4. Fine-tuned model shows adapted geometry for cultural understanding")

# ============================================================================
# CELL 18: Export Results (Optional)
# ============================================================================
print("\n" + "="*80)
print("EXPORTING RESULTS")
print("="*80)

results_dict = {
    "model_name": BASE_MODEL_NAME,
    "finetuned_path": FINETUNED_MODEL_PATH,
    "num_layers": num_layers,
    "vocab_size": vocab_size,
    "datasets": {
        "squad_samples": len(squad_dataset),
        "cultural_samples": len(cultural_dataset)
    },
    "spectral_curvature": {
        "squad_mean": squad_curv_mean.tolist() if len(squad_curv_mean) > 0 else [],
        "cultural_mean": cultural_curv_mean.tolist() if len(cultural_curv_mean) > 0 else []
    },
    "thermodynamic_length": {
        "parameter_based": squad_param_thermo.tolist(),
        "prediction_based": squad_pred_thermo.tolist()
    },
    "belief_vectors": {
        "squad": squad_belief.tolist(),
        "cultural_by_country": {k: v.tolist() for k, v in cultural_belief_results.items()}
    },
    "configuration": {
        "max_seq_len": MAX_SEQ_LEN,
        "batch_size": BATCH_SIZE,
        "tau": TAU,
        "fr_norm": FR_NORM
    }
}

import json
output_file = "ndna_analysis_results.json"
with open(output_file, 'w') as f:
    json.dump(results_dict, f, indent=2)

print(f"✅ Results exported to: {output_file}")
print("\n🎉 Analysis complete! All plots generated and results saved.")

In [ ]:
# ============================================================================
# COMPLETE nDNA ANALYSIS FOR CULTURALLY FINE-TUNED LLAMA MODEL (ERROR-FREE)
# ============================================================================
# This notebook performs nDNA geometric analysis on your fine-tuned model
# Metrics: Spectral Curvature, Thermodynamic Length, Belief Vector Fields
# Datasets: SQuAD v2 and CulturalBench
# Hardware: Optimized for Google Colab T4 GPU
# ============================================================================

# ============================================================================
# CELL 1: Fix NumPy Compatibility Issue
# ============================================================================
import sys
print("🔧 Fixing NumPy compatibility issue...")

# Uninstall conflicting packages and reinstall with correct versions
!pip uninstall -y numpy -q
!pip uninstall -y transformers datasets accelerate peft torch matplotlib scipy plotly -q
!pip install numpy==1.24.3 -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install transformers==4.36.0 datasets==2.16.0 accelerate==0.25.0 peft==0.7.1 -q
!pip install matplotlib plotly scipy scikit-learn -q

print("✅ Packages installed successfully!")

# ============================================================================
# CELL 2: Restart Runtime Warning
# ============================================================================
print("\n" + "="*80)
print("⚠️  IMPORTANT: After running Cell 1, you may need to restart the runtime.")
print("   In Colab: Runtime → Restart runtime")
print("   Then run from Cell 3 onwards.")
print("="*80)

# ============================================================================
# CELL 3: Imports (Run after restart if needed)
# ============================================================================
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

import math
import random
from tqdm import tqdm
from functools import partial
import warnings
warnings.filterwarnings('ignore')

print("✅ All packages imported successfully!")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🔢 NumPy version: {np.__version__}")
if torch.cuda.is_available():
    print(f"🖥️  GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected!")

# ============================================================================
# CELL 4: Configuration
# ============================================================================
print("\n" + "="*80)
print("CONFIGURATION FOR nDNA ANALYSIS")
print("="*80)

# Model paths - MODIFY THESE TO YOUR PATHS
BASE_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
FINETUNED_MODEL_PATH = "./llama-3.2-3b-finetuned-CulturalBench"  # Your local path

# Device configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

# Dataset configuration
MAX_SAMPLES_SQUAD = 100          # Reduced for faster testing
MAX_SAMPLES_CULTURAL = 100       # Reduced for faster testing
BATCH_SIZE = 1                   # Small batch for T4
MAX_SEQ_LEN = 256               # Moderate sequence length

# nDNA Analysis parameters
TOKENS_PER_EXAMPLE = 32         # For belief vectors
TAU = 1.0                       # Temperature for softmax
FR_NORM = True                  # Fisher-Rao normalization
EPS_DIST = 1e-12               # Numerical stability
EPS_CURV = 1e-12               # Curvature epsilon

# Mixed precision
AMP_DTYPE = (torch.bfloat16 if (DEVICE=="cuda" and torch.cuda.is_bf16_supported())
             else torch.float16)

print(f"✅ Device: {DEVICE}")
print(f"✅ AMP dtype: {AMP_DTYPE}")
print(f"✅ Base model: {BASE_MODEL_NAME}")
print(f"✅ Fine-tuned model: {FINETUNED_MODEL_PATH}")

# ============================================================================
# CELL 5: Set Random Seeds
# ============================================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("✅ Random seeds set for reproducibility")

# ============================================================================
# CELL 6: Load Fine-Tuned Model
# ============================================================================
print("\n" + "="*80)
print("LOADING FINE-TUNED MODEL")
print("="*80)

try:
    print(f"Loading tokenizer from {BASE_MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "right"

    print(f"Loading base model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

    print(f"Loading LoRA adapters from {FINETUNED_MODEL_PATH}...")
    model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL_PATH)
    model.eval()

    # Get model components (LLaMA architecture)
    llama_model = model.base_model.model.model  # Access the base LlamaModel
    blocks = llama_model.layers                  # Decoder layers
    final_norm = llama_model.norm               # Final RMSNorm
    lm_head = model.base_model.model.lm_head   # Language modeling head

    num_layers = len(blocks)
    vocab_size = model.config.vocab_size

    print(f"✅ Model loaded successfully!")
    print(f"   Layers: {num_layers}")
    print(f"   Vocabulary size: {vocab_size}")

except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Ensure FINETUNED_MODEL_PATH exists and contains adapter_model.bin")
    print("   2. Check that you have access to meta-llama/Llama-3.2-3B-Instruct")
    print("   3. Verify you're logged into Hugging Face Hub")
    raise

# ============================================================================
# CELL 7: Load Datasets
# ============================================================================
print("\n" + "="*80)
print("LOADING DATASETS")
print("="*80)

# ===== SQuAD v2 Dataset =====
print("\n📚 Loading SQuAD v2...")
try:
    squad_raw = load_dataset("squad_v2", split="train")

    def format_squad(example):
        """Format SQuAD example as prompt"""
        question = example["question"].strip()
        context = example["context"].strip()
        return {"text": f"Question: {question}\nContext: {context}\nAnswer:"}

    squad_dataset = squad_raw.map(format_squad, remove_columns=squad_raw.column_names)
    squad_dataset = squad_dataset.filter(lambda ex: len(ex["text"]) > 50)
    if MAX_SAMPLES_SQUAD:
        squad_dataset = squad_dataset.select(range(min(MAX_SAMPLES_SQUAD, len(squad_dataset))))

    print(f"✅ SQuAD v2: {len(squad_dataset)} samples")
    print(f"   Sample: {squad_dataset[0]['text'][:150]}...")

except Exception as e:
    print(f"❌ Error loading SQuAD v2: {e}")
    print("Creating fallback dataset...")
    # Create a small fallback dataset
    squad_dataset = Dataset.from_dict({
        "text": [
            "Question: What is machine learning?\nContext: Machine learning is a subset of artificial intelligence.\nAnswer:",
            "Question: What is deep learning?\nContext: Deep learning uses neural networks with multiple layers.\nAnswer:"
        ]
    })
    print(f"✅ Using fallback dataset with {len(squad_dataset)} samples")

# ===== CulturalBench Dataset =====
print("\n🌍 Loading CulturalBench...")
try:
    cultural_raw = load_dataset("kellycyy/CulturalBench", "CulturalBench-Hard")

    def format_cultural(example):
        """Format CulturalBench example"""
        question = str(example.get("prompt_question", "")).strip()
        options = str(example.get("prompt_option", "")).strip()
        return {"text": f"{question} {options}".strip(), "country": example.get("country", "Unknown")}

    cultural_dataset = cultural_raw["test"].map(
        format_cultural,
        remove_columns=[col for col in cultural_raw["test"].column_names if col not in ["country"]]
    )
    cultural_dataset = cultural_dataset.filter(lambda ex: len(ex["text"]) > 20)

    # Group by country for analysis
    countries = list(set(cultural_dataset["country"]))
    cultural_by_country = {
        country: cultural_dataset.filter(lambda ex: ex["country"] == country)
        for country in countries[:3]  # Top 3 countries for visualization
    }

    print(f"✅ CulturalBench: {len(cultural_dataset)} samples")
    print(f"   Countries: {len(countries)}")
    print(f"   Top countries: {list(cultural_by_country.keys())}")
    print(f"   Sample: {cultural_dataset[0]['text'][:150]}...")

except Exception as e:
    print(f"❌ Error loading CulturalBench: {e}")
    print("Creating fallback dataset...")
    # Create a small fallback dataset
    cultural_dataset = Dataset.from_dict({
        "text": [
            "What is the significance of Diwali in Indian culture?",
            "Explain the concept of ubuntu in African philosophy."
        ],
        "country": ["India", "South Africa"]
    })
    cultural_by_country = {"India": cultural_dataset}
    print(f"✅ Using fallback dataset with {len(cultural_dataset)} samples")

# ============================================================================
# CELL 8: Data Collation Functions
# ============================================================================
print("\n" + "="*80)
print("SETTING UP DATA COLLATORS")
print("="*80)

def causal_collate_standard(batch):
    """Standard collation for thermodynamic length (full sequence)"""
    texts = [ex["text"] for ex in batch]
    tok = tokenizer(
        texts,
        padding="longest",
        truncation=True,
        max_length=MAX_SEQ_LEN,
        return_tensors="pt"
    )

    input_ids = tok["input_ids"]
    attention_mask = tok["attention_mask"]

    # Shift for next-token prediction
    x = input_ids[:, :-1].contiguous()
    y = input_ids[:, 1:].contiguous()
    attn = attention_mask[:, :-1].contiguous()

    # Mask padding in labels
    y = y.masked_fill(attention_mask[:, 1:] == 0, -100)

    return {
        "input_ids": x,
        "attention_mask": attn,
        "labels": y
    }

def causal_collate_belief(batch, keep_last_k=TOKENS_PER_EXAMPLE):
    """Collation for belief vectors (keep last K tokens)"""
    texts = [ex["text"] for ex in batch]
    tok = tokenizer(
        texts,
        padding="longest",
        truncation=True,
        max_length=MAX_SEQ_LEN,
        return_tensors="pt"
    )

    input_ids = tok["input_ids"]
    attention_mask = tok["attention_mask"]

    x = input_ids[:, :-1].contiguous()
    y = input_ids[:, 1:].contiguous()
    attn = attention_mask[:, :-1].contiguous()
    y = y.masked_fill(attention_mask[:, 1:] == 0, -100)

    # Select mask: keep only last K supervised positions
    B, S = x.shape
    sel_mask = torch.zeros_like(y, dtype=torch.bool)
    for b in range(B):
        valid = (y[b] != -100).nonzero(as_tuple=False).squeeze(-1)
        if valid.numel() > 0:
            take = valid[-min(keep_last_k, valid.numel()):]
            sel_mask[b, take] = True

    return {
        "input_ids": x,
        "attention_mask": attn,
        "labels": y,
        "select_mask": sel_mask
    }

print("✅ Data collators configured")

# ============================================================================
# CELL 9: Geometry Helper Functions
# ============================================================================
print("\n" + "="*80)
print("DEFINING GEOMETRY HELPERS")
print("="*80)

def sqrt_embed(q: torch.Tensor, eps: float = EPS_DIST) -> torch.Tensor:
    """Map probability distribution q to unit sphere via sqrt embedding"""
    q = torch.clamp(q, min=eps)
    q = q / q.sum()
    u = torch.sqrt(q)
    u = u / (torch.norm(u, p=2) + 1e-30)
    return u

def project_tangent(u: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Project vector v onto tangent space at u on the sphere"""
    return v - torch.dot(u, v) * u

def fisher_rao_distance(logp1: torch.Tensor, logp2: torch.Tensor) -> torch.Tensor:
    """Compute Fisher-Rao distance between two log-probability distributions"""
    s = 0.5 * (logp1 + logp2)
    log_bc = torch.logsumexp(s, dim=-1)
    bc = torch.exp(log_bc).clamp(0.0, 1.0)
    return 2.0 * torch.acos(bc)

print("✅ Geometry helpers defined")

# ============================================================================
# CELL 10: METRIC 1 - Spectral Curvature
# ============================================================================
print("\n" + "="*80)
print("METRIC 1: SPECTRAL CURVATURE")
print("="*80)

@torch.no_grad()
def compute_spectral_curvature(text: str):
    """Compute spectral curvature across layers using logit lens"""
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=True).to(DEVICE)
    if enc.input_ids.shape[1] == 0:
        raise ValueError("Empty input")

    input_ids = enc.input_ids
    attention_mask = enc.attention_mask

    # Forward pass to get all hidden states
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states

    # Get last position
    t = input_ids.shape[1] - 1

    # Extract distributions at each layer
    q_list = []
    for h in hidden_states:
        vec = h[0, t, :]
        vec_norm = final_norm(vec)
        logits = lm_head(vec_norm)
        q = torch.softmax(logits.float() / TAU, dim=-1)
        q_list.append(q)

    # Map to sphere
    u_list = [sqrt_embed(q) for q in q_list]

    # Compute curvature
    m = len(u_list)
    if m < 3:
        return np.array([]), np.array([])

    # First differences
    delta_u = []
    speeds = []
    for ell in range(m - 1):
        u_curr = u_list[ell]
        v = u_list[ell + 1] - u_curr
        du = project_tangent(u_curr, v)
        delta_u.append(du)
        speeds.append(torch.norm(du, p=2).item())

    # Second differences (curvatures)
    curvatures = []
    for ell in range(1, m - 1):
        u_curr = u_list[ell]
        v2 = u_list[ell + 1] - 2 * u_list[ell] + u_list[ell - 1]
        d2u = project_tangent(u_curr, v2)

        num = torch.norm(d2u, p=2)
        s = torch.norm(delta_u[ell], p=2)
        denom = (s * s + EPS_CURV) ** 1.5
        kappa = (num / denom).item()
        curvatures.append(kappa)

    return np.array(curvatures), np.array(speeds)

# Test spectral curvature
print("\n🧪 Testing Spectral Curvature...")
try:
    test_text = squad_dataset[0]["text"]
    curvs, speeds = compute_spectral_curvature(test_text)
    print(f"✅ Computed {len(curvs)} curvature values")
    if len(curvs) > 0:
        print(f"   Mean curvature: {curvs.mean():.6e}")
except Exception as e:
    print(f"❌ Error in spectral curvature: {e}")

# ============================================================================
# CELL 11: METRIC 2 - Prediction-based Thermodynamic Length
# ============================================================================
print("\n" + "="*80)
print("METRIC 2: THERMODYNAMIC LENGTH (PREDICTION-BASED)")
print("="*80)

@torch.no_grad()
def compute_prediction_thermodynamic_length(dataset, dataset_name: str):
    """Compute Fisher-Rao distance between consecutive layer predictions"""
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=causal_collate_standard
    )

    num_steps = num_layers - 1
    fr_step_sums = torch.zeros(num_steps, device=DEVICE)
    fr_step_counts = torch.zeros(num_steps, device=DEVICE)

    print(f"\n🔥 Computing for {dataset_name}...")

    for batch in tqdm(loader, desc=f"[{dataset_name}]"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        B, S = input_ids.shape

        # Get initial embeddings
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
            h = llama_model.embed_tokens(input_ids)

        logp_prev = None

        for ell in range(num_layers):
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
                layer_output = blocks[ell](h, attention_mask=attention_mask)
                h = layer_output[0]

                logits = lm_head(final_norm(h))
                logp = F.log_softmax(logits.float(), dim=-1)

            if logp_prev is not None:
                valid = (labels != -100)
                fr_dists = fisher_rao_distance(logp_prev, logp)

                step_idx = ell - 1
                fr_step_sums[step_idx] += fr_dists.masked_fill(~valid, 0.0).sum()
                fr_step_counts[step_idx] += valid.sum()

            logp_prev = logp

    fr_step_means = (fr_step_sums / fr_step_counts.clamp_min(1)).cpu().numpy()
    print(f"✅ Computed prediction-based thermodynamic length")
    return fr_step_means

# Compute for SQuAD
try:
    squad_pred_thermo = compute_prediction_thermodynamic_length(squad_dataset, "SQuAD v2")
except Exception as e:
    print(f"❌ Error: {e}")
    squad_pred_thermo = np.zeros(num_layers - 1)

# ============================================================================
# CELL 12: METRIC 3 - Belief Vector Fields
# ============================================================================
print("\n" + "="*80)
print("METRIC 3: BELIEF VECTOR FIELDS")
print("="*80)

@torch.no_grad()
def compute_belief_vectors(dataset, dataset_name: str):
    """Compute belief vector field magnitudes across layers"""
    loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        collate_fn=causal_collate_belief
    )

    v_sum = [torch.zeros(vocab_size, device=DEVICE, dtype=torch.float32) for _ in range(num_layers)]
    u_sum = [torch.zeros(vocab_size, device=DEVICE, dtype=torch.float32) for _ in range(num_layers)]
    cnt = [0 for _ in range(num_layers)]

    print(f"\n🔥 Computing for {dataset_name}...")

    for batch in tqdm(loader, desc=f"[{dataset_name}]"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        keep_mask = batch["select_mask"].to(DEVICE)

        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
                output_hidden_states=True
            )
            hidden_states = outputs.hidden_states

        for ell in range(num_layers):
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
                h_ell = hidden_states[ell + 1]
                z = lm_head(final_norm(h_ell))

            keep = keep_mask.view(-1)
            if not keep.any():
                continue

            z_sel = z.view(-1, vocab_size)[keep]
            y_sel = labels.view(-1)[keep]

            z32 = z_sel.float() / TAU
            q = F.softmax(z32, dim=-1)
            q = torch.clamp(q, min=1e-12)
            u = torch.sqrt(q)

            g = -q / TAU
            add = torch.full((y_sel.shape[0], 1), 1.0/TAU, device=DEVICE, dtype=q.dtype)
            g.scatter_add_(dim=-1, index=y_sel.unsqueeze(-1), src=add)

            ug = u * g
            s = (q * g).sum(dim=-1, keepdim=True)
            t = (ug - s * u) / (2.0 * TAU)

            v_sum[ell] += t.sum(dim=0).to(v_sum[ell].dtype)
            u_sum[ell] += u.sum(dim=0).to(u_sum[ell].dtype)
            cnt[ell] += t.size(0)

    norms = []
    for ell in range(num_layers):
        if cnt[ell] == 0:
            norms.append(0.0)
            continue

        v_avg = v_sum[ell] / cnt[ell]
        u_avg = u_sum[ell] / cnt[ell]

        u_norm = torch.linalg.norm(u_avg).clamp_min(1e-12)
        u_bar = u_avg / u_norm

        radial = torch.dot(u_bar, v_avg)
        v_tan = v_avg - radial * u_bar

        n = torch.linalg.norm(v_tan).item()
        norms.append(2.0 * n if FR_NORM else n)

    print(f"✅ Computed belief vectors")
    return np.array(norms)

# Compute for SQuAD
try:
    squad_belief = compute_belief_vectors(squad_dataset, "SQuAD v2")
except Exception as e:
    print(f"❌ Error: {e}")
    squad_belief = np.zeros(num_layers)

# Compute for cultural data
cultural_belief_results = {}
try:
    for country in list(cultural_by_country.keys())[:2]:
        ds = cultural_by_country[country]
        if len(ds) > MAX_SAMPLES_CULTURAL:
            ds = ds.select(range(MAX_SAMPLES_CULTURAL))
        belief_norms = compute_belief_vectors(ds, f"Cultural-{country}")
        cultural_belief_results[country] = belief_norms
except Exception as e:
    print(f"❌ Error in cultural belief vectors: {e}")

# ============================================================================
# CELL 13: Visualization - All Metrics
# ============================================================================
print("\n" + "="*80)
print("GENERATING VISUALIZATIONS")
print("="*80)

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle('nDNA Analysis - Fine-tuned Llama Model', fontsize=16, fontweight='bold')

# Plot 1: Thermodynamic Length (Prediction-based)
ax = axes[0]
if len(squad_pred_thermo) > 0:
    steps = np.arange(1, len(squad_pred_thermo) + 1)
    ax.plot(steps, squad_pred_thermo, marker='^', color='blue', linewidth=2, label='SQuAD v2')
ax.set_title('Thermodynamic Length (Fisher-Rao Distance)', fontsize=13, fontweight='bold')
ax.set_xlabel('Layer Transition (ℓ → ℓ+1)', fontsize=11)
ax.set_ylabel('Mean FR Distance (radians)', fontsize=11)
ax.grid(True, alpha=0.3)
ax.legend()

# Plot 2: Belief Vectors - SQuAD
ax = axes[1]
if len(squad_belief) > 0:
    layers = np.arange(1, len(squad_belief) + 1)
    ax.plot(layers, squad_belief, marker='o', color='blue', linewidth=2, label='SQuAD v2')
ax.set_title('Belief Vector Field Magnitudes', fontsize=13, fontweight='bold')
ax.set_xlabel('Layer Index', fontsize=11)
ax.set_ylabel('||v_ℓ|| (Fisher-Rao norm)', fontsize=11)
ax.grid(True, alpha=0.3)
ax.legend()

# Plot 3: Belief Vectors - Cultural (if available)
ax = axes[2]
colors = ['green', 'orange', 'purple']
if cultural_belief_results:
    for i, (country, norms) in enumerate(cultural_belief_results.items()):
        layers = np.arange(1, len(norms) + 1)
        ax.plot(layers, norms, marker='s', color=colors[i % len(colors)],
                linewidth=2, label=country, markersize=6)
ax.set_title('Belief Vectors - CulturalBench', fontsize=13, fontweight='bold')
ax.set_xlabel('Layer Index', fontsize=11)
ax.set_ylabel('||v_ℓ|| (Fisher-Rao norm)', fontsize=11)
ax.grid(True, alpha=0.3)
if cultural_belief_results:
    ax.legend()

plt.tight_layout()
plt.show()

print("✅ Visualizations complete!")

# ============================================================================
# CELL 14: Summary Statistics
# ============================================================================
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print("\n📏 THERMODYNAMIC LENGTH (Prediction-based)")
print("-" * 60)
if len(squad_pred_thermo) > 0:
    print(f"SQuAD v2:")
    print(f"  Total FR distance: {squad_pred_thermo.sum():.6e} radians")
    print(f"  Mean per step: {squad_pred_thermo.mean():.6e} radians")
    print(f"  Max step: {squad_pred_thermo.max():.6e}")

print("\n🎯 BELIEF VECTORS")
print("-" * 60)
if len(squad_belief) > 0:
    print(f"SQuAD v2:")
    print(f"  Mean magnitude: {squad_belief.mean():.6e}")
    print(f"  Max magnitude: {squad_belief.max():.6e}")
    print(f"  Std deviation: {squad_belief.std():.6e}")

if cultural_belief_results:
    print(f"\nCulturalBench:")
    for country, norms in cultural_belief_results.items():
        print(f"  {country}: mean={norms.mean():.6e}, max={norms.max():.6e}")

print("\n" + "="*80)
print("✅ nDNA ANALYSIS COMPLETE!")
print("="*80)

In [ ]:
# ============================================================================
# COMPLETE nDNA ANALYSIS FOR CULTURALLY FINE-TUNED LLAMA MODEL (ERROR-FREE)
# ============================================================================
# This notebook performs nDNA geometric analysis on your fine-tuned model
# Metrics: Spectral Curvature, Thermodynamic Length, Belief Vector Fields
# Datasets: SQuAD v2 and CulturalBench
# Hardware: Optimized for Google Colab T4 GPU
# ============================================================================

# ============================================================================
# CELL 1: Fix NumPy Compatibility Issue
# ============================================================================
import sys
print("🔧 Fixing NumPy compatibility issue...")

# Uninstall conflicting packages and reinstall with correct versions
!pip uninstall -y numpy -q
!pip uninstall -y transformers datasets accelerate peft torch matplotlib scipy plotly -q
!pip install numpy==1.24.3 -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install transformers==4.36.0 datasets==2.16.0 accelerate==0.25.0 peft==0.7.1 -q
!pip install matplotlib plotly scipy scikit-learn -q

print("✅ Packages installed successfully!")

# ============================================================================
# CELL 2: Restart Runtime Warning
# ============================================================================
print("\n" + "="*80)
print("⚠️  IMPORTANT: After running Cell 1, you may need to restart the runtime.")
print("   In Colab: Runtime → Restart runtime")
print("   Then run from Cell 3 onwards.")
print("="*80)

# ============================================================================
# CELL 3: Imports (Run after restart if needed)
# ============================================================================
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

import math
import random
from tqdm import tqdm
from functools import partial
import warnings
warnings.filterwarnings('ignore')

print("✅ All packages imported successfully!")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🔢 NumPy version: {np.__version__}")
if torch.cuda.is_available():
    print(f"🖥️  GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected!")

# ============================================================================
# CELL 4: Configuration
# ============================================================================
print("\n" + "="*80)
print("CONFIGURATION FOR nDNA ANALYSIS")
print("="*80)

# Model paths - MODIFY THESE TO YOUR PATHS
BASE_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
FINETUNED_MODEL_PATH = "./llama-3.2-3b-finetuned-CulturalBench"  # Your local path

# Device configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

# Dataset configuration
MAX_SAMPLES_SQUAD = 100          # Reduced for faster testing
MAX_SAMPLES_CULTURAL = 100       # Reduced for faster testing
BATCH_SIZE = 1                   # Small batch for T4
MAX_SEQ_LEN = 256               # Moderate sequence length

# nDNA Analysis parameters
TOKENS_PER_EXAMPLE = 32         # For belief vectors
TAU = 1.0                       # Temperature for softmax
FR_NORM = True                  # Fisher-Rao normalization
EPS_DIST = 1e-12               # Numerical stability
EPS_CURV = 1e-12               # Curvature epsilon

# Mixed precision
AMP_DTYPE = (torch.bfloat16 if (DEVICE=="cuda" and torch.cuda.is_bf16_supported())
             else torch.float16)

print(f"✅ Device: {DEVICE}")
print(f"✅ AMP dtype: {AMP_DTYPE}")
print(f"✅ Base model: {BASE_MODEL_NAME}")
print(f"✅ Fine-tuned model: {FINETUNED_MODEL_PATH}")

# ============================================================================
# CELL 5: Set Random Seeds
# ============================================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("✅ Random seeds set for reproducibility")

# ============================================================================
# CELL 6: Load Fine-Tuned Model
# ============================================================================
print("\n" + "="*80)
print("LOADING FINE-TUNED MODEL")
print("="*80)

try:
    print(f"Loading tokenizer from {BASE_MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "right"

    print(f"Loading base model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

    print(f"Loading LoRA adapters from {FINETUNED_MODEL_PATH}...")
    model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL_PATH)
    model.eval()

    # Get model components (LLaMA architecture)
    llama_model = model.base_model.model.model  # Access the base LlamaModel
    blocks = llama_model.layers                  # Decoder layers
    final_norm = llama_model.norm               # Final RMSNorm
    lm_head = model.base_model.model.lm_head   # Language modeling head

    num_layers = len(blocks)
    vocab_size = model.config.vocab_size

    print(f"✅ Model loaded successfully!")
    print(f"   Layers: {num_layers}")
    print(f"   Vocabulary size: {vocab_size}")

except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Ensure FINETUNED_MODEL_PATH exists and contains adapter_model.bin")
    print("   2. Check that you have access to meta-llama/Llama-3.2-3B-Instruct")
    print("   3. Verify you're logged into Hugging Face Hub")
    raise

# ============================================================================
# CELL 7: Load Datasets
# ============================================================================
print("\n" + "="*80)
print("LOADING DATASETS")
print("="*80)

# ===== SQuAD v2 Dataset =====
print("\n📚 Loading SQuAD v2...")
try:
    squad_raw = load_dataset("squad_v2", split="train")

    def format_squad(example):
        """Format SQuAD example as prompt"""
        question = example["question"].strip()
        context = example["context"].strip()
        return {"text": f"Question: {question}\nContext: {context}\nAnswer:"}

    squad_dataset = squad_raw.map(format_squad, remove_columns=squad_raw.column_names)
    squad_dataset = squad_dataset.filter(lambda ex: len(ex["text"]) > 50)
    if MAX_SAMPLES_SQUAD:
        squad_dataset = squad_dataset.select(range(min(MAX_SAMPLES_SQUAD, len(squad_dataset))))

    print(f"✅ SQuAD v2: {len(squad_dataset)} samples")
    print(f"   Sample: {squad_dataset[0]['text'][:150]}...")

except Exception as e:
    print(f"❌ Error loading SQuAD v2: {e}")
    print("Creating fallback dataset...")
    # Create a small fallback dataset
    squad_dataset = Dataset.from_dict({
        "text": [
            "Question: What is machine learning?\nContext: Machine learning is a subset of artificial intelligence.\nAnswer:",
            "Question: What is deep learning?\nContext: Deep learning uses neural networks with multiple layers.\nAnswer:"
        ]
    })
    print(f"✅ Using fallback dataset with {len(squad_dataset)} samples")

# ===== CulturalBench Dataset =====
print("\n🌍 Loading CulturalBench...")
try:
    cultural_raw = load_dataset("kellycyy/CulturalBench", "CulturalBench-Hard")

    def format_cultural(example):
        """Format CulturalBench example"""
        question = str(example.get("prompt_question", "")).strip()
        options = str(example.get("prompt_option", "")).strip()
        return {"text": f"{question} {options}".strip(), "country": example.get("country", "Unknown")}

    cultural_dataset = cultural_raw["test"].map(
        format_cultural,
        remove_columns=[col for col in cultural_raw["test"].column_names if col not in ["country"]]
    )
    cultural_dataset = cultural_dataset.filter(lambda ex: len(ex["text"]) > 20)

    # Group by country for analysis
    countries = list(set(cultural_dataset["country"]))
    cultural_by_country = {
        country: cultural_dataset.filter(lambda ex: ex["country"] == country)
        for country in countries[:3]  # Top 3 countries for visualization
    }

    print(f"✅ CulturalBench: {len(cultural_dataset)} samples")
    print(f"   Countries: {len(countries)}")
    print(f"   Top countries: {list(cultural_by_country.keys())}")
    print(f"   Sample: {cultural_dataset[0]['text'][:150]}...")

except Exception as e:
    print(f"❌ Error loading CulturalBench: {e}")
    print("Creating fallback dataset...")
    # Create a small fallback dataset
    cultural_dataset = Dataset.from_dict({
        "text": [
            "What is the significance of Diwali in Indian culture?",
            "Explain the concept of ubuntu in African philosophy."
        ],
        "country": ["India", "South Africa"]
    })
    cultural_by_country = {"India": cultural_dataset}
    print(f"✅ Using fallback dataset with {len(cultural_dataset)} samples")

# ============================================================================
# CELL 8: Data Collation Functions
# ============================================================================
print("\n" + "="*80)
print("SETTING UP DATA COLLATORS")
print("="*80)

def causal_collate_standard(batch):
    """Standard collation for thermodynamic length (full sequence)"""
    texts = [ex["text"] for ex in batch]
    tok = tokenizer(
        texts,
        padding="longest",
        truncation=True,
        max_length=MAX_SEQ_LEN,
        return_tensors="pt"
    )

    input_ids = tok["input_ids"]
    attention_mask = tok["attention_mask"]

    # Shift for next-token prediction
    x = input_ids[:, :-1].contiguous()
    y = input_ids[:, 1:].contiguous()
    attn = attention_mask[:, :-1].contiguous()

    # Mask padding in labels
    y = y.masked_fill(attention_mask[:, 1:] == 0, -100)

    return {
        "input_ids": x,
        "attention_mask": attn,
        "labels": y
    }

def causal_collate_belief(batch, keep_last_k=TOKENS_PER_EXAMPLE):
    """Collation for belief vectors (keep last K tokens)"""
    texts = [ex["text"] for ex in batch]
    tok = tokenizer(
        texts,
        padding="longest",
        truncation=True,
        max_length=MAX_SEQ_LEN,
        return_tensors="pt"
    )

    input_ids = tok["input_ids"]
    attention_mask = tok["attention_mask"]

    x = input_ids[:, :-1].contiguous()
    y = input_ids[:, 1:].contiguous()
    attn = attention_mask[:, :-1].contiguous()
    y = y.masked_fill(attention_mask[:, 1:] == 0, -100)

    # Select mask: keep only last K supervised positions
    B, S = x.shape
    sel_mask = torch.zeros_like(y, dtype=torch.bool)
    for b in range(B):
        valid = (y[b] != -100).nonzero(as_tuple=False).squeeze(-1)
        if valid.numel() > 0:
            take = valid[-min(keep_last_k, valid.numel()):]
            sel_mask[b, take] = True

    return {
        "input_ids": x,
        "attention_mask": attn,
        "labels": y,
        "select_mask": sel_mask
    }

print("✅ Data collators configured")

# ============================================================================
# CELL 9: Geometry Helper Functions
# ============================================================================
print("\n" + "="*80)
print("DEFINING GEOMETRY HELPERS")
print("="*80)

def sqrt_embed(q: torch.Tensor, eps: float = EPS_DIST) -> torch.Tensor:
    """Map probability distribution q to unit sphere via sqrt embedding"""
    q = torch.clamp(q, min=eps)
    q = q / q.sum()
    u = torch.sqrt(q)
    u = u / (torch.norm(u, p=2) + 1e-30)
    return u

def project_tangent(u: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Project vector v onto tangent space at u on the sphere"""
    return v - torch.dot(u, v) * u

def fisher_rao_distance(logp1: torch.Tensor, logp2: torch.Tensor) -> torch.Tensor:
    """Compute Fisher-Rao distance between two log-probability distributions"""
    s = 0.5 * (logp1 + logp2)
    log_bc = torch.logsumexp(s, dim=-1)
    bc = torch.exp(log_bc).clamp(0.0, 1.0)
    return 2.0 * torch.acos(bc)

print("✅ Geometry helpers defined")

# ============================================================================
# CELL 10: METRIC 1 - Spectral Curvature
# ============================================================================
print("\n" + "="*80)
print("METRIC 1: SPECTRAL CURVATURE")
print("="*80)

@torch.no_grad()
def compute_spectral_curvature(text: str):
    """Compute spectral curvature across layers using logit lens"""
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=True).to(DEVICE)
    if enc.input_ids.shape[1] == 0:
        raise ValueError("Empty input")

    input_ids = enc.input_ids
    attention_mask = enc.attention_mask

    # Forward pass to get all hidden states
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states

    # Get last position
    t = input_ids.shape[1] - 1

    # Extract distributions at each layer
    q_list = []
    for h in hidden_states:
        vec = h[0, t, :]
        vec_norm = final_norm(vec)
        logits = lm_head(vec_norm)
        q = torch.softmax(logits.float() / TAU, dim=-1)
        q_list.append(q)

    # Map to sphere
    u_list = [sqrt_embed(q) for q in q_list]

    # Compute curvature
    m = len(u_list)
    if m < 3:
        return np.array([]), np.array([])

    # First differences
    delta_u = []
    speeds = []
    for ell in range(m - 1):
        u_curr = u_list[ell]
        v = u_list[ell + 1] - u_curr
        du = project_tangent(u_curr, v)
        delta_u.append(du)
        speeds.append(torch.norm(du, p=2).item())

    # Second differences (curvatures)
    curvatures = []
    for ell in range(1, m - 1):
        u_curr = u_list[ell]
        v2 = u_list[ell + 1] - 2 * u_list[ell] + u_list[ell - 1]
        d2u = project_tangent(u_curr, v2)

        num = torch.norm(d2u, p=2)
        s = torch.norm(delta_u[ell], p=2)
        denom = (s * s + EPS_CURV) ** 1.5
        kappa = (num / denom).item()
        curvatures.append(kappa)

    return np.array(curvatures), np.array(speeds)

# Test spectral curvature
print("\n🧪 Testing Spectral Curvature...")
try:
    test_text = squad_dataset[0]["text"]
    curvs, speeds = compute_spectral_curvature(test_text)
    print(f"✅ Computed {len(curvs)} curvature values")
    if len(curvs) > 0:
        print(f"   Mean curvature: {curvs.mean():.6e}")
except Exception as e:
    print(f"❌ Error in spectral curvature: {e}")

# ============================================================================
# CELL 11: METRIC 2 - Prediction-based Thermodynamic Length
# ============================================================================
print("\n" + "="*80)
print("METRIC 2: THERMODYNAMIC LENGTH (PREDICTION-BASED)")
print("="*80)

@torch.no_grad()
def compute_prediction_thermodynamic_length(dataset, dataset_name: str):
    """Compute Fisher-Rao distance between consecutive layer predictions"""
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=causal_collate_standard
    )

    num_steps = num_layers - 1
    fr_step_sums = torch.zeros(num_steps, device=DEVICE)
    fr_step_counts = torch.zeros(num_steps, device=DEVICE)

    print(f"\n🔥 Computing for {dataset_name}...")

    for batch in tqdm(loader, desc=f"[{dataset_name}]"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        B, S = input_ids.shape

        # Get initial embeddings
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
            h = llama_model.embed_tokens(input_ids)

        logp_prev = None

        for ell in range(num_layers):
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
                layer_output = blocks[ell](h, attention_mask=attention_mask)
                h = layer_output[0]

                logits = lm_head(final_norm(h))
                logp = F.log_softmax(logits.float(), dim=-1)

            if logp_prev is not None:
                valid = (labels != -100)
                fr_dists = fisher_rao_distance(logp_prev, logp)

                step_idx = ell - 1
                fr_step_sums[step_idx] += fr_dists.masked_fill(~valid, 0.0).sum()
                fr_step_counts[step_idx] += valid.sum()

            logp_prev = logp

    fr_step_means = (fr_step_sums / fr_step_counts.clamp_min(1)).cpu().numpy()
    print(f"✅ Computed prediction-based thermodynamic length")
    return fr_step_means

# Compute for SQuAD
try:
    squad_pred_thermo = compute_prediction_thermodynamic_length(squad_dataset, "SQuAD v2")
except Exception as e:
    print(f"❌ Error: {e}")
    squad_pred_thermo = np.zeros(num_layers - 1)

# ============================================================================
# CELL 12: METRIC 3 - Belief Vector Fields
# ============================================================================
print("\n" + "="*80)
print("METRIC 3: BELIEF VECTOR FIELDS")
print("="*80)

@torch.no_grad()
def compute_belief_vectors(dataset, dataset_name: str):
    """Compute belief vector field magnitudes across layers"""
    loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        collate_fn=causal_collate_belief
    )

    v_sum = [torch.zeros(vocab_size, device=DEVICE, dtype=torch.float32) for _ in range(num_layers)]
    u_sum = [torch.zeros(vocab_size, device=DEVICE, dtype=torch.float32) for _ in range(num_layers)]
    cnt = [0 for _ in range(num_layers)]

    print(f"\n🔥 Computing for {dataset_name}...")

    for batch in tqdm(loader, desc=f"[{dataset_name}]"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        keep_mask = batch["select_mask"].to(DEVICE)

        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
                output_hidden_states=True
            )
            hidden_states = outputs.hidden_states

        for ell in range(num_layers):
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
                h_ell = hidden_states[ell + 1]
                z = lm_head(final_norm(h_ell))

            keep = keep_mask.view(-1)
            if not keep.any():
                continue

            z_sel = z.view(-1, vocab_size)[keep]
            y_sel = labels.view(-1)[keep]

            z32 = z_sel.float() / TAU
            q = F.softmax(z32, dim=-1)
            q = torch.clamp(q, min=1e-12)
            u = torch.sqrt(q)

            g = -q / TAU
            add = torch.full((y_sel.shape[0], 1), 1.0/TAU, device=DEVICE, dtype=q.dtype)
            g.scatter_add_(dim=-1, index=y_sel.unsqueeze(-1), src=add)

            ug = u * g
            s = (q * g).sum(dim=-1, keepdim=True)
            t = (ug - s * u) / (2.0 * TAU)

            v_sum[ell] += t.sum(dim=0).to(v_sum[ell].dtype)
            u_sum[ell] += u.sum(dim=0).to(u_sum[ell].dtype)
            cnt[ell] += t.size(0)

    norms = []
    for ell in range(num_layers):
        if cnt[ell] == 0:
            norms.append(0.0)
            continue

        v_avg = v_sum[ell] / cnt[ell]
        u_avg = u_sum[ell] / cnt[ell]

        u_norm = torch.linalg.norm(u_avg).clamp_min(1e-12)
        u_bar = u_avg / u_norm

        radial = torch.dot(u_bar, v_avg)
        v_tan = v_avg - radial * u_bar

        n = torch.linalg.norm(v_tan).item()
        norms.append(2.0 * n if FR_NORM else n)

    print(f"✅ Computed belief vectors")
    return np.array(norms)

# Compute for SQuAD
try:
    squad_belief = compute_belief_vectors(squad_dataset, "SQuAD v2")
except Exception as e:
    print(f"❌ Error: {e}")
    squad_belief = np.zeros(num_layers)

# Compute for cultural data
cultural_belief_results = {}
try:
    for country in list(cultural_by_country.keys())[:2]:
        ds = cultural_by_country[country]
        if len(ds) > MAX_SAMPLES_CULTURAL:
            ds = ds.select(range(MAX_SAMPLES_CULTURAL))
        belief_norms = compute_belief_vectors(ds, f"Cultural-{country}")
        cultural_belief_results[country] = belief_norms
except Exception as e:
    print(f"❌ Error in cultural belief vectors: {e}")

# ============================================================================
# CELL 13: Visualization - All Metrics
# ============================================================================
print("\n" + "="*80)
print("GENERATING VISUALIZATIONS")
print("="*80)

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle('nDNA Analysis - Fine-tuned Llama Model', fontsize=16, fontweight='bold')

# Plot 1: Thermodynamic Length (Prediction-based)
ax = axes[0]
if len(squad_pred_thermo) > 0:
    steps = np.arange(1, len(squad_pred_thermo) + 1)
    ax.plot(steps, squad_pred_thermo, marker='^', color='blue', linewidth=2, label='SQuAD v2')
ax.set_title('Thermodynamic Length (Fisher-Rao Distance)', fontsize=13, fontweight='bold')
ax.set_xlabel('Layer Transition (ℓ → ℓ+1)', fontsize=11)
ax.set_ylabel('Mean FR Distance (radians)', fontsize=11)
ax.grid(True, alpha=0.3)
ax.legend()

# Plot 2: Belief Vectors - SQuAD
ax = axes[1]
if len(squad_belief) > 0:
    layers = np.arange(1, len(squad_belief) + 1)
    ax.plot(layers, squad_belief, marker='o', color='blue', linewidth=2, label='SQuAD v2')
ax.set_title('Belief Vector Field Magnitudes', fontsize=13, fontweight='bold')
ax.set_xlabel('Layer Index', fontsize=11)
ax.set_ylabel('||v_ℓ|| (Fisher-Rao norm)', fontsize=11)
ax.grid(True, alpha=0.3)
ax.legend()

# Plot 3: Belief Vectors - Cultural (if available)
ax = axes[2]
colors = ['green', 'orange', 'purple']
if cultural_belief_results:
    for i, (country, norms) in enumerate(cultural_belief_results.items()):
        layers = np.arange(1, len(norms) + 1)
        ax.plot(layers, norms, marker='s', color=colors[i % len(colors)],
                linewidth=2, label=country, markersize=6)
ax.set_title('Belief Vectors - CulturalBench', fontsize=13, fontweight='bold')
ax.set_xlabel('Layer Index', fontsize=11)
ax.set_ylabel('||v_ℓ|| (Fisher-Rao norm)', fontsize=11)
ax.grid(True, alpha=0.3)
if cultural_belief_results:
    ax.legend()

plt.tight_layout()
plt.show()

print("✅ Visualizations complete!")

# ============================================================================
# CELL 14: Summary Statistics
# ============================================================================
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print("\n📏 THERMODYNAMIC LENGTH (Prediction-based)")
print("-" * 60)
if len(squad_pred_thermo) > 0:
    print(f"SQuAD v2:")
    print(f"  Total FR distance: {squad_pred_thermo.sum():.6e} radians")
    print(f"  Mean per step: {squad_pred_thermo.mean():.6e} radians")
    print(f"  Max step: {squad_pred_thermo.max():.6e}")

print("\n🎯 BELIEF VECTORS")
print("-" * 60)
if len(squad_belief) > 0:
    print(f"SQuAD v2:")
    print(f"  Mean magnitude: {squad_belief.mean():.6e}")
    print(f"  Max magnitude: {squad_belief.max():.6e}")
    print(f"  Std deviation: {squad_belief.std():.6e}")

if cultural_belief_results:
    print(f"\nCulturalBench:")
    for country, norms in cultural_belief_results.items():
        print(f"  {country}: mean={norms.mean():.6e}, max={norms.max():.6e}")

print("\n" + "="*80)
print("✅ nDNA ANALYSIS COMPLETE!")
print("="*80)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
import gc
import peft.PeftModel

warnings.filterwarnings('ignore')



# Update this path to match your unzipped folder
MODEL_PATH = "/content/llama-3.2-3b-finetuned-CulturalBench"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Model path: {MODEL_PATH}")
print(f"Device: {DEVICE}\n")

# ============================================================================
# STEP 2: LOAD YOUR FINE-TUNED MODEL (NO BASE MODEL, NO PRETRAINED)
# ============================================================================

print("="*80)
print("LOADING YOUR FINE-TUNED MODEL")
print("="*80 + "\n")

try:
    from transformers import AutoTokenizer, AutoModelForCausalLM


    # Load tokenizer from your saved model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print(f"✓ Tokenizer loaded (vocab size: {tokenizer.vocab_size})\n")

    # Try loading as PEFT model first (since you used LoRA)
    try:
        print("Attempting to load as PEFT model...")
        base_model = AutoModelForCausalLM.from_pretrained(
            "meta-llama/Llama-3.2-3B-Instruct",  # Base architecture needed for PEFT
            torch_dtype=torch.float16,
            device_map="auto",
            low_cpu_mem_usage=True
        )
        model = PeftModel.from_pretrained(base_model, MODEL_PATH)
        model = model.merge_and_unload()  # Merge LoRA into base
        print("✓ PEFT model loaded and merged\n")
    except:
        print("Loading as standard model...")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            torch_dtype=torch.float16,
            device_map="auto",
            low_cpu_mem_usage=True
        )
        print("✓ Model loaded\n")

    model.eval()

    # Verify model structure
    if hasattr(model, 'model') and hasattr(model.model, 'norm'):
        final_norm = model.model.norm
        print(f"✓ Final layer norm: {type(final_norm).__name__}")
    else:
        final_norm = None
        print(f"⚠ No final layer norm (will use identity)")

    if hasattr(model, 'lm_head'):
        print(f"✓ LM head: {type(model.lm_head).__name__}")

    num_layers = model.config.num_hidden_layers if hasattr(model.config, 'num_hidden_layers') else 28
    vocab_size = model.config.vocab_size if hasattr(model.config, 'vocab_size') else 128256

    print(f"✓ Layers: {num_layers}")
    print(f"✓ Vocab: {vocab_size}\n")

except Exception as e:
    print(f"❌ Error loading model: {e}")
    raise

# ============================================================================
# STEP 3: LOAD DATASETS (SQuAD v2 + CulturalBench)
# ============================================================================

print("="*80)
print("LOADING EVALUATION DATASETS")
print("="*80 + "\n")

from datasets import load_dataset

texts_squad = []
texts_cultural = []

# Load SQuAD v2
try:
    print("Loading SQuAD v2 validation set...")
    squad = load_dataset("rajpurkar/squad_v2", split="validation[:15]")
    for sample in squad:
        context = sample["context"]
        if len(context) > 50:
            texts_squad.append(context)
    texts_squad = texts_squad[:10]
    print(f"✓ Loaded {len(texts_squad)} SQuAD v2 contexts\n")
except Exception as e:
    print(f"⚠ Could not load SQuAD v2: {e}")
    texts_squad = [
        "Machine learning enables systems to learn from experience without explicit programming.",
        "Deep learning uses neural networks with multiple layers to learn representations.",
        "Natural language processing deals with interactions between computers and human language.",
        "Question answering systems automatically respond to questions using knowledge bases.",
        "Transformers have revolutionized NLP through self-attention mechanisms."
    ]
    print(f"✓ Using {len(texts_squad)} fallback SQuAD texts\n")

# Load CulturalBench
try:
    print("Loading CulturalBench dataset...")
    cultural = load_dataset("kellycyy/CulturalBench", "CulturalBench-Hard", split="test[:15]")
    for sample in cultural:
        q = str(sample.get("prompt_question", ""))
        opt = str(sample.get("prompt_option", ""))
        ans = str(sample.get("answer", ""))
        combined_text = f"{q} {opt} {ans}".strip()
        if len(combined_text) > 50:
            texts_cultural.append(combined_text)
    texts_cultural = texts_cultural[:10]
    print(f"✓ Loaded {len(texts_cultural)} CulturalBench samples\n")
except Exception as e:
    print(f"⚠ Could not load CulturalBench: {e}")
    texts_cultural = [
        "Cultural context influences language understanding and generation patterns.",
        "Different regions have distinct linguistic features and communication styles.",
        "Cross-cultural communication requires awareness of cultural nuances.",
        "Language models must account for diverse cultural perspectives.",
        "Cultural sensitivity is essential for global AI applications."
    ]
    print(f"✓ Using {len(texts_cultural)} fallback cultural texts\n")

# Combine datasets
all_texts = texts_squad + texts_cultural
dataset_labels = (["SQuAD v2"] * len(texts_squad)) + (["CulturalBench"] * len(texts_cultural))

print(f"✓ Total samples for analysis: {len(all_texts)}\n")

# ============================================================================
# HELPER FUNCTIONS: GEOMETRIC OPERATIONS (OFFICIAL nDNA)
# ============================================================================

def project_tangent(u, v, eps=1e-12):
    """
    Project vector v onto tangent space at u on unit sphere.
    Formula: v_tangent = v - <u,v> * u
    """
    dot = (u * v).sum(dim=-1, keepdim=True)
    return v - dot * u

def sqrt_embed(q, eps=1e-12):
    """
    Convert probability distribution q to unit-norm sphere embedding.
    Formula: u = sqrt(q) / ||sqrt(q)||
    Maps simplex to sphere for geometric operations.
    """
    q = torch.clamp(q, min=eps)
    u = torch.sqrt(q)
    u = u / (u.norm(dim=-1, keepdim=True) + eps)
    return u

def safe_apply_final_norm(hidden_state):
    """Safely apply final layer norm if available."""
    try:
        if final_norm is not None:
            return final_norm(hidden_state)
        else:
            return hidden_state
    except:
        return hidden_state

def safe_get_logits(hidden_state):
    """Safely extract logits from hidden state."""
    try:
        h_norm = safe_apply_final_norm(hidden_state)
        logits = model.lm_head(h_norm)
        return logits
    except:
        try:
            logits = model.lm_head(hidden_state)
            return logits
        except:
            return torch.zeros(vocab_size, device=hidden_state.device)

# ============================================================================
# nDNA METRIC 1: SPECTRAL CURVATURE (κ)
# ============================================================================

def compute_spectral_curvature(ulist, eps_curv=1e-12):
    """
    Compute spectral curvature κ per layer.

    Formula (Official nDNA):
    κ_ℓ = ||d²u_ℓ|| / (||du_{ℓ-1}|| × ||du_ℓ||)

    Interpretation: Nonlinearity of representation trajectory
    """
    m = len(ulist)
    if m < 3:
        return np.array([])

    # First differences
    delta_u = []
    for ell in range(m - 1):
        u_curr = ulist[ell]
        u_next = ulist[ell + 1]
        du = project_tangent(u_curr, u_next - u_curr)
        delta_u.append(du)

    # Curvature at interior points
    kappas = []
    for ell in range(1, m - 1):
        try:
            u_prev = ulist[ell - 1]
            u_curr = ulist[ell]
            u_next = ulist[ell + 1]

            # Second difference
            d2u_raw = u_next - 2 * u_curr + u_prev
            d2u = project_tangent(u_curr, d2u_raw)

            # Curvature formula
            num = d2u.norm(p=2)
            s_prev = delta_u[ell - 1].norm(p=2)
            s_curr = delta_u[ell].norm(p=2)
            denom = s_prev * s_curr + eps_curv

            kappa = (num / denom).item()
            kappas.append(max(kappa, 0.0))
        except:
            kappas.append(0.0)

    return np.array(kappas)

# ============================================================================
# nDNA METRIC 2: THERMODYNAMIC LENGTH (L) - FISHER-RAO
# ============================================================================

def compute_fisher_rao_distance(logp_prev, logp_curr, eps=1e-12):
    """
    Compute Fisher-Rao geodesic distance.

    Formula (Official nDNA):
    L = 2 × arccos(BC)
    where BC = Σ_i sqrt(p_i × q_i)

    Interpretation: Information-theoretic distance
    """
    try:
        p_prev = torch.exp(torch.clamp(logp_prev, max=10, min=-100))
        p_curr = torch.exp(torch.clamp(logp_curr, max=10, min=-100))

        p_prev = torch.clamp(p_prev, min=eps, max=1.0)
        p_curr = torch.clamp(p_curr, min=eps, max=1.0)

        # Bhattacharyya coefficient
        sqrt_product = torch.sqrt(p_prev * p_curr)
        bc = torch.sum(sqrt_product)
        bc = torch.clamp(bc, 0.0, 1.0)

        # Fisher-Rao distance
        fr_dist = 2.0 * torch.acos(bc)

        return fr_dist.item()
    except:
        return 0.0

# ============================================================================
# nDNA METRIC 3: BELIEF VECTOR (Δθ)
# ============================================================================

def compute_belief_vector_change(logp_prev, logp_curr):
    """
    Compute belief vector change.

    Formula (Official nDNA):
    ||Δθ|| = ||log p_{ℓ+1} - log p_ℓ||_2

    Interpretation: Magnitude of belief shift
    """
    try:
        logp_prev = torch.clamp(logp_prev, min=-100, max=10)
        logp_curr = torch.clamp(logp_curr, min=-100, max=10)

        delta_theta = logp_curr - logp_prev
        delta_norm = delta_theta.norm(p=2).item()

        return delta_norm
    except:
        return 0.0

# ============================================================================
# MAIN COMPUTATION: PROCESS ALL SAMPLES
# ============================================================================

print("="*80)
print("COMPUTING nDNA METRICS")
print("="*80 + "\n")

all_kappas_squad = []
all_fr_squad = []
all_belief_squad = []

all_kappas_cultural = []
all_fr_cultural = []
all_belief_cultural = []

model.eval()

with torch.no_grad():
    for sample_idx, (text, dataset_label) in enumerate(tqdm(zip(all_texts, dataset_labels), total=len(all_texts), desc="Processing samples")):
        try:
            # Tokenize
            enc = tokenizer(
                text,
                return_tensors="pt",
                max_length=256,
                truncation=True,
                padding=True
            ).to(DEVICE)

            if enc['input_ids'].shape[1] < 10:
                continue

            # Forward pass
            outputs = model(
                enc['input_ids'],
                attention_mask=enc.get('attention_mask'),
                output_hidden_states=True
            )

            hidden_states = outputs.hidden_states
            if hidden_states is None or len(hidden_states) == 0:
                continue

            seq_len = enc['input_ids'].shape[1]

            # Extract representations at each layer
            ulist = []
            logp_list = []

            for h in hidden_states:
                try:
                    h_last = h[0, seq_len - 1, :]
                    logits = safe_get_logits(h_last)

                    if torch.isnan(logits).any() or torch.isinf(logits).any():
                        continue

                    logp = F.log_softmax(logits, dim=-1)
                    p = torch.exp(torch.clamp(logp, max=0))
                    u = sqrt_embed(p)

                    ulist.append(u)
                    logp_list.append(logp)
                except:
                    continue

            if len(ulist) < 3:
                continue

            # Compute metrics
            kappas = compute_spectral_curvature(ulist)

            fr_dists = []
            belief_changes = []

            for ell in range(len(logp_list) - 1):
                try:
                    logp_prev = logp_list[ell]
                    logp_curr = logp_list[ell + 1]

                    fr = compute_fisher_rao_distance(logp_prev, logp_curr)
                    if not np.isnan(fr) and not np.isinf(fr):
                        fr_dists.append(fr)

                    belief = compute_belief_vector_change(logp_prev, logp_curr)
                    if not np.isnan(belief) and not np.isinf(belief):
                        belief_changes.append(belief)
                except:
                    continue

            # Store by dataset
            if dataset_label == "SQuAD v2":
                if len(kappas) > 0:
                    all_kappas_squad.append(kappas)
                if len(fr_dists) > 0:
                    all_fr_squad.append(fr_dists)
                if len(belief_changes) > 0:
                    all_belief_squad.append(belief_changes)
            else:  # CulturalBench
                if len(kappas) > 0:
                    all_kappas_cultural.append(kappas)
                if len(fr_dists) > 0:
                    all_fr_cultural.append(fr_dists)
                if len(belief_changes) > 0:
                    all_belief_cultural.append(belief_changes)

            # Clear memory
            del outputs, hidden_states, enc
            torch.cuda.empty_cache()
            gc.collect()

        except Exception as e:
            continue

print("\n✓ nDNA metric computation complete!\n")

# ============================================================================
# AGGREGATE METRICS
# ============================================================================

print("Aggregating metrics...\n")

def aggregate_metrics(metric_list):
    if not metric_list:
        return None
    max_len = max(len(m) for m in metric_list)
    padded = np.zeros((len(metric_list), max_len))
    for i, m in enumerate(metric_list):
        padded[i, :len(m)] = m
    return np.mean(padded, axis=0)

mean_kappas_squad = aggregate_metrics(all_kappas_squad)
mean_fr_squad = aggregate_metrics(all_fr_squad)
mean_belief_squad = aggregate_metrics(all_belief_squad)

mean_kappas_cultural = aggregate_metrics(all_kappas_cultural)
mean_fr_cultural = aggregate_metrics(all_fr_cultural)
mean_belief_cultural = aggregate_metrics(all_belief_cultural)

print(f"SQuAD v2 - Spectral: {len(mean_kappas_squad) if mean_kappas_squad is not None else 0} layers")
print(f"CulturalBench - Spectral: {len(mean_kappas_cultural) if mean_kappas_cultural is not None else 0} layers\n")

# ============================================================================
# VISUALIZATION: DUAL-DATASET COMPARISON PLOTS
# ============================================================================

print("="*80)
print("GENERATING PUBLICATION-QUALITY PLOTS")
print("="*80 + "\n")

if (mean_kappas_squad is not None and mean_kappas_cultural is not None and
    mean_fr_squad is not None and mean_fr_cultural is not None and
    mean_belief_squad is not None and mean_belief_cultural is not None):

    fig, axes = plt.subplots(3, 1, figsize=(15, 14))
    fig.suptitle(
        "nDNA Analysis: Culturally Fine-Tuned LLaMA 3.2\n" +
        "Comparative Analysis: SQuAD v2 vs CulturalBench",
        fontsize=17, fontweight='bold'
    )

    # ---- PLOT 1: SPECTRAL CURVATURE ----
    ax = axes[0]

    layers_squad = np.arange(2, len(mean_kappas_squad) + 2)
    layers_cultural = np.arange(2, len(mean_kappas_cultural) + 2)

    ax.plot(layers_squad, mean_kappas_squad, 'o-', lw=2.5, ms=7,
            color='#3498db', label='SQuAD v2', alpha=0.8)
    ax.fill_between(layers_squad, mean_kappas_squad, alpha=0.15, color='#3498db')

    ax.plot(layers_cultural, mean_kappas_cultural, 's-', lw=2.5, ms=7,
            color='#e74c3c', label='CulturalBench', alpha=0.8)
    ax.fill_between(layers_cultural, mean_kappas_cultural, alpha=0.15, color='#e74c3c')

    ax.set_xlabel('Layer Index', fontsize=12, fontweight='bold')
    ax.set_ylabel('Spectral Curvature κ', fontsize=12, fontweight='bold')
    ax.set_title(
        'Metric 1: Spectral Curvature (Per Layer)\n' +
        'Interpretation: Nonlinearity of representation trajectory',
        fontsize=12, fontweight='bold'
    )
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=11, loc='best')
    ax.set_facecolor('#f8f9fa')

    # ---- PLOT 2: THERMODYNAMIC LENGTH ----
    ax = axes[1]

    transitions_squad = np.arange(1, len(mean_fr_squad) + 1)
    transitions_cultural = np.arange(1, len(mean_fr_cultural) + 1)

    ax.plot(transitions_squad, mean_fr_squad, 'o-', lw=2.5, ms=7,
            color='#3498db', label='SQuAD v2', alpha=0.8)
    ax.fill_between(transitions_squad, mean_fr_squad, alpha=0.15, color='#3498db')

    ax.plot(transitions_cultural, mean_fr_cultural, 's-', lw=2.5, ms=7,
            color='#e74c3c', label='CulturalBench', alpha=0.8)
    ax.fill_between(transitions_cultural, mean_fr_cultural, alpha=0.15, color='#e74c3c')

    ax.set_xlabel('Layer Transition (ℓ → ℓ+1)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Thermodynamic Length L (radians)', fontsize=12, fontweight='bold')
    ax.set_title(
        'Metric 2: Thermodynamic Length (Fisher-Rao Distance)\n' +
        'Interpretation: Information-theoretic distance between layers',
        fontsize=12, fontweight='bold'
    )
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=11, loc='best')
    ax.set_facecolor('#f8f9fa')

    # ---- PLOT 3: BELIEF VECTOR ----
    ax = axes[2]

    transitions_squad = np.arange(1, len(mean_belief_squad) + 1)
    transitions_cultural = np.arange(1, len(mean_belief_cultural) + 1)

    ax.plot(transitions_squad, mean_belief_squad, 'o-', lw=2.5, ms=7,
            color='#3498db', label='SQuAD v2', alpha=0.8)
    ax.fill_between(transitions_squad, mean_belief_squad, alpha=0.15, color='#3498db')

    ax.plot(transitions_cultural, mean_belief_cultural, 's-', lw=2.5, ms=7,
            color='#e74c3c', label='CulturalBench', alpha=0.8)
    ax.fill_between(transitions_cultural, mean_belief_cultural, alpha=0.15, color='#e74c3c')

    # Add mean lines
    mean_squad = np.mean(mean_belief_squad)
    mean_cultural = np.mean(mean_belief_cultural)
    ax.axhline(mean_squad, linestyle=':', lw=2, color='#2980b9', alpha=0.6,
               label=f'SQuAD mean={mean_squad:.4f}')
    ax.axhline(mean_cultural, linestyle=':', lw=2, color='#c0392b', alpha=0.6,
               label=f'Cultural mean={mean_cultural:.4f}')

    ax.set_xlabel('Layer Transition (ℓ → ℓ+1)', fontsize=12, fontweight='bold')
    ax.set_ylabel('||Δθ|| (L2 Norm)', fontsize=12, fontweight='bold')
    ax.set_title(
        'Metric 3: Belief Vector Change\n' +
        'Interpretation: Magnitude of belief shift in natural parameter space',
        fontsize=12, fontweight='bold'
    )
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=10, loc='best')
    ax.set_facecolor('#f8f9fa')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig('nDNA_cultural_llama_dual_dataset.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("✓ Plot saved as 'nDNA_cultural_llama_dual_dataset.png'\n")

# ============================================================================
# SUMMARY REPORT
# ============================================================================

print("="*80)
print("nDNA ANALYSIS SUMMARY REPORT")
print("="*80 + "\n")

print(f"Model: YOUR LOCALLY SAVED CULTURALLY FINE-TUNED LLAMA 3.2")
print(f"Path: {MODEL_PATH}")
print(f"Datasets: SQuAD v2 + CulturalBench")
print(f"Total Samples: {len(all_texts)}")
print(f"Model Layers: {num_layers}\n")

print("="*80)
print("SQuAD v2 RESULTS")
print("="*80 + "\n")

if mean_kappas_squad is not None:
    print(f"✓ Spectral Curvature (κ)")
    print(f"  Mean: {np.mean(mean_kappas_squad):.6f}")
    print(f"  Std:  {np.std(mean_kappas_squad):.6f}\n")

if mean_fr_squad is not None:
    print(f"✓ Thermodynamic Length (L)")
    print(f"  Mean: {np.mean(mean_fr_squad):.6f}")
    print(f"  Std:  {np.std(mean_fr_squad):.6f}\n")

if mean_belief_squad is not None:
    print(f"✓ Belief Vector (Δθ)")
    print(f"  Mean: {np.mean(mean_belief_squad):.6f}")
    print(f"  Std:  {np.std(mean_belief_squad):.6f}\n")

print("="*80)
print("CULTURALBENCH RESULTS")
print("="*80 + "\n")

if mean_kappas_cultural is not None:
    print(f"✓ Spectral Curvature (κ)")
    print(f"  Mean: {np.mean(mean_kappas_cultural):.6f}")
    print(f"  Std:  {np.std(mean_kappas_cultural):.6f}\n")

if mean_fr_cultural is not None:
    print(f"✓ Thermodynamic Length (L)")
    print(f"  Mean: {np.mean(mean_fr_cultural):.6f}")
    print(f"  Std:  {np.std(mean_fr_cultural):.6f}\n")

if mean_belief_cultural is not None:
    print(f"✓ Belief Vector (Δθ)")
    print(f"  Mean: {np.mean(mean_belief_cultural):.6f}")
    print(f"  Std:  {np.std(mean_belief_cultural):.6f}\n")

print("="*80)
print("✅ nDNA ANALYSIS COMPLETE")
print("="*80)
print("\nOutput:")
print("  • nDNA_cultural_llama_dual_dataset.png")
print("\nMethodology:")
print("  ✓ Official nDNA formulas (SPINAL/pragyaai)")
print("  ✓ Fisher-Rao geodesic geometry")
print("  ✓ Tangent space projections")
print("  ✓ Your culturally fine-tuned model ONLY")
print("  ✓ Dual dataset comparison (SQuAD v2 + CulturalBench)")
print("  ✓ T4 GPU optimized\n")


In [ ]:
# ============================================================================
# COMPLETE nDNA ANALYSIS FOR CULTURALLY FINE-TUNED LLAMA MODEL (ERROR-FREE)
# ============================================================================
# This notebook performs nDNA geometric analysis on your fine-tuned model
# Metrics: Spectral Curvature, Thermodynamic Length, Belief Vector Fields
# Datasets: SQuAD v2 and CulturalBench
# Hardware: Optimized for Google Colab T4 GPU
# ============================================================================

# ============================================================================
# CELL 1: Fix NumPy Compatibility Issue
# ============================================================================
import sys
print("🔧 Fixing NumPy compatibility issue...")

# Uninstall conflicting packages and reinstall with correct versions
!pip uninstall -y numpy -q
!pip uninstall -y transformers datasets accelerate peft torch matplotlib scipy plotly -q
!pip install numpy==1.24.3 -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install transformers==4.36.0 datasets==2.16.0 accelerate==0.25.0 peft==0.7.1 -q
!pip install matplotlib plotly scipy scikit-learn -q

print("✅ Packages installed successfully!")

# ============================================================================
# CELL 2: Restart Runtime Warning
# ============================================================================
print("\n" + "="*80)
print("⚠️  IMPORTANT: After running Cell 1, you may need to restart the runtime.")
print("   In Colab: Runtime → Restart runtime")
print("   Then run from Cell 3 onwards.")
print("="*80)

# ============================================================================
# CELL 3: Imports (Run after restart if needed)
# ============================================================================
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

import math
import random
from tqdm import tqdm
from functools import partial
import warnings
warnings.filterwarnings('ignore')

print("✅ All packages imported successfully!")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🔢 NumPy version: {np.__version__}")
if torch.cuda.is_available():
    print(f"🖥️  GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected!")

# ============================================================================
# CELL 4: Configuration
# ============================================================================
print("\n" + "="*80)
print("CONFIGURATION FOR nDNA ANALYSIS")
print("="*80)

# Model paths - MODIFY THESE TO YOUR PATHS
BASE_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
FINETUNED_MODEL_PATH = "./llama-3.2-3b-finetuned-CulturalBench"  # Your local path

# Device configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

# Dataset configuration
MAX_SAMPLES_SQUAD = 100          # Reduced for faster testing
MAX_SAMPLES_CULTURAL = 100       # Reduced for faster testing
BATCH_SIZE = 1                   # Small batch for T4
MAX_SEQ_LEN = 256               # Moderate sequence length

# nDNA Analysis parameters
TOKENS_PER_EXAMPLE = 32         # For belief vectors
TAU = 1.0                       # Temperature for softmax
FR_NORM = True                  # Fisher-Rao normalization
EPS_DIST = 1e-12               # Numerical stability
EPS_CURV = 1e-12               # Curvature epsilon

# Mixed precision
AMP_DTYPE = (torch.bfloat16 if (DEVICE=="cuda" and torch.cuda.is_bf16_supported())
             else torch.float16)

print(f"✅ Device: {DEVICE}")
print(f"✅ AMP dtype: {AMP_DTYPE}")
print(f"✅ Base model: {BASE_MODEL_NAME}")
print(f"✅ Fine-tuned model: {FINETUNED_MODEL_PATH}")

# ============================================================================
# CELL 5: Set Random Seeds
# ============================================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("✅ Random seeds set for reproducibility")

# ============================================================================
# CELL 6: Load Fine-Tuned Model
# ============================================================================
print("\n" + "="*80)
print("LOADING FINE-TUNED MODEL")
print("="*80)

try:
    print(f"Loading tokenizer from {BASE_MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "right"

    print(f"Loading base model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

    print(f"Loading LoRA adapters from {FINETUNED_MODEL_PATH}...")
    model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL_PATH)
    model.eval()

    # Get model components (LLaMA architecture)
    llama_model = model.base_model.model.model  # Access the base LlamaModel
    blocks = llama_model.layers                  # Decoder layers
    final_norm = llama_model.norm               # Final RMSNorm
    lm_head = model.base_model.model.lm_head   # Language modeling head

    num_layers = len(blocks)
    vocab_size = model.config.vocab_size

    print(f"✅ Model loaded successfully!")
    print(f"   Layers: {num_layers}")
    print(f"   Vocabulary size: {vocab_size}")

except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Ensure FINETUNED_MODEL_PATH exists and contains adapter_model.bin")
    print("   2. Check that you have access to meta-llama/Llama-3.2-3B-Instruct")
    print("   3. Verify you're logged into Hugging Face Hub")
    raise

# ============================================================================
# CELL 7: Load Datasets
# ============================================================================
print("\n" + "="*80)
print("LOADING DATASETS")
print("="*80)

# ===== SQuAD v2 Dataset =====
print("\n📚 Loading SQuAD v2...")
try:
    squad_raw = load_dataset("squad_v2", split="train")

    def format_squad(example):
        """Format SQuAD example as prompt"""
        question = example["question"].strip()
        context = example["context"].strip()
        return {"text": f"Question: {question}\nContext: {context}\nAnswer:"}

    squad_dataset = squad_raw.map(format_squad, remove_columns=squad_raw.column_names)
    squad_dataset = squad_dataset.filter(lambda ex: len(ex["text"]) > 50)
    if MAX_SAMPLES_SQUAD:
        squad_dataset = squad_dataset.select(range(min(MAX_SAMPLES_SQUAD, len(squad_dataset))))

    print(f"✅ SQuAD v2: {len(squad_dataset)} samples")
    print(f"   Sample: {squad_dataset[0]['text'][:150]}...")

except Exception as e:
    print(f"❌ Error loading SQuAD v2: {e}")
    print("Creating fallback dataset...")
    # Create a small fallback dataset
    squad_dataset = Dataset.from_dict({
        "text": [
            "Question: What is machine learning?\nContext: Machine learning is a subset of artificial intelligence.\nAnswer:",
            "Question: What is deep learning?\nContext: Deep learning uses neural networks with multiple layers.\nAnswer:"
        ]
    })
    print(f"✅ Using fallback dataset with {len(squad_dataset)} samples")

# ===== CulturalBench Dataset =====
print("\n🌍 Loading CulturalBench...")
try:
    cultural_raw = load_dataset("kellycyy/CulturalBench", "CulturalBench-Hard")

    def format_cultural(example):
        """Format CulturalBench example"""
        question = str(example.get("prompt_question", "")).strip()
        options = str(example.get("prompt_option", "")).strip()
        return {"text": f"{question} {options}".strip(), "country": example.get("country", "Unknown")}

    cultural_dataset = cultural_raw["test"].map(
        format_cultural,
        remove_columns=[col for col in cultural_raw["test"].column_names if col not in ["country"]]
    )
    cultural_dataset = cultural_dataset.filter(lambda ex: len(ex["text"]) > 20)

    # Group by country for analysis
    countries = list(set(cultural_dataset["country"]))
    cultural_by_country = {
        country: cultural_dataset.filter(lambda ex: ex["country"] == country)
        for country in countries[:3]  # Top 3 countries for visualization
    }

    print(f"✅ CulturalBench: {len(cultural_dataset)} samples")
    print(f"   Countries: {len(countries)}")
    print(f"   Top countries: {list(cultural_by_country.keys())}")
    print(f"   Sample: {cultural_dataset[0]['text'][:150]}...")

except Exception as e:
    print(f"❌ Error loading CulturalBench: {e}")
    print("Creating fallback dataset...")
    # Create a small fallback dataset
    cultural_dataset = Dataset.from_dict({
        "text": [
            "What is the significance of Diwali in Indian culture?",
            "Explain the concept of ubuntu in African philosophy."
        ],
        "country": ["India", "South Africa"]
    })
    cultural_by_country = {"India": cultural_dataset}
    print(f"✅ Using fallback dataset with {len(cultural_dataset)} samples")

# ============================================================================
# CELL 8: Data Collation Functions
# ============================================================================
print("\n" + "="*80)
print("SETTING UP DATA COLLATORS")
print("="*80)

def causal_collate_standard(batch):
    """Standard collation for thermodynamic length (full sequence)"""
    texts = [ex["text"] for ex in batch]
    tok = tokenizer(
        texts,
        padding="longest",
        truncation=True,
        max_length=MAX_SEQ_LEN,
        return_tensors="pt"
    )

    input_ids = tok["input_ids"]
    attention_mask = tok["attention_mask"]

    # Shift for next-token prediction
    x = input_ids[:, :-1].contiguous()
    y = input_ids[:, 1:].contiguous()
    attn = attention_mask[:, :-1].contiguous()

    # Mask padding in labels
    y = y.masked_fill(attention_mask[:, 1:] == 0, -100)

    return {
        "input_ids": x,
        "attention_mask": attn,
        "labels": y
    }

def causal_collate_belief(batch, keep_last_k=TOKENS_PER_EXAMPLE):
    """Collation for belief vectors (keep last K tokens)"""
    texts = [ex["text"] for ex in batch]
    tok = tokenizer(
        texts,
        padding="longest",
        truncation=True,
        max_length=MAX_SEQ_LEN,
        return_tensors="pt"
    )

    input_ids = tok["input_ids"]
    attention_mask = tok["attention_mask"]

    x = input_ids[:, :-1].contiguous()
    y = input_ids[:, 1:].contiguous()
    attn = attention_mask[:, :-1].contiguous()
    y = y.masked_fill(attention_mask[:, 1:] == 0, -100)

    # Select mask: keep only last K supervised positions
    B, S = x.shape
    sel_mask = torch.zeros_like(y, dtype=torch.bool)
    for b in range(B):
        valid = (y[b] != -100).nonzero(as_tuple=False).squeeze(-1)
        if valid.numel() > 0:
            take = valid[-min(keep_last_k, valid.numel()):]
            sel_mask[b, take] = True

    return {
        "input_ids": x,
        "attention_mask": attn,
        "labels": y,
        "select_mask": sel_mask
    }

print("✅ Data collators configured")

# ============================================================================
# CELL 9: Geometry Helper Functions
# ============================================================================
print("\n" + "="*80)
print("DEFINING GEOMETRY HELPERS")
print("="*80)

def sqrt_embed(q: torch.Tensor, eps: float = EPS_DIST) -> torch.Tensor:
    """Map probability distribution q to unit sphere via sqrt embedding"""
    q = torch.clamp(q, min=eps)
    q = q / q.sum()
    u = torch.sqrt(q)
    u = u / (torch.norm(u, p=2) + 1e-30)
    return u

def project_tangent(u: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Project vector v onto tangent space at u on the sphere"""
    return v - torch.dot(u, v) * u

def fisher_rao_distance(logp1: torch.Tensor, logp2: torch.Tensor) -> torch.Tensor:
    """Compute Fisher-Rao distance between two log-probability distributions"""
    s = 0.5 * (logp1 + logp2)
    log_bc = torch.logsumexp(s, dim=-1)
    bc = torch.exp(log_bc).clamp(0.0, 1.0)
    return 2.0 * torch.acos(bc)

print("✅ Geometry helpers defined")

# ============================================================================
# CELL 10: METRIC 1 - Spectral Curvature
# ============================================================================
print("\n" + "="*80)
print("METRIC 1: SPECTRAL CURVATURE")
print("="*80)

@torch.no_grad()
def compute_spectral_curvature(text: str):
    """Compute spectral curvature across layers using logit lens"""
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=True).to(DEVICE)
    if enc.input_ids.shape[1] == 0:
        raise ValueError("Empty input")

    input_ids = enc.input_ids
    attention_mask = enc.attention_mask

    # Forward pass to get all hidden states
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states

    # Get last position
    t = input_ids.shape[1] - 1

    # Extract distributions at each layer
    q_list = []
    for h in hidden_states:
        vec = h[0, t, :]
        vec_norm = final_norm(vec)
        logits = lm_head(vec_norm)
        q = torch.softmax(logits.float() / TAU, dim=-1)
        q_list.append(q)

    # Map to sphere
    u_list = [sqrt_embed(q) for q in q_list]

    # Compute curvature
    m = len(u_list)
    if m < 3:
        return np.array([]), np.array([])

    # First differences
    delta_u = []
    speeds = []
    for ell in range(m - 1):
        u_curr = u_list[ell]
        v = u_list[ell + 1] - u_curr
        du = project_tangent(u_curr, v)
        delta_u.append(du)
        speeds.append(torch.norm(du, p=2).item())

    # Second differences (curvatures)
    curvatures = []
    for ell in range(1, m - 1):
        u_curr = u_list[ell]
        v2 = u_list[ell + 1] - 2 * u_list[ell] + u_list[ell - 1]
        d2u = project_tangent(u_curr, v2)

        num = torch.norm(d2u, p=2)
        s = torch.norm(delta_u[ell], p=2)
        denom = (s * s + EPS_CURV) ** 1.5
        kappa = (num / denom).item()
        curvatures.append(kappa)

    return np.array(curvatures), np.array(speeds)

# Test spectral curvature
print("\n🧪 Testing Spectral Curvature...")
try:
    test_text = squad_dataset[0]["text"]
    curvs, speeds = compute_spectral_curvature(test_text)
    print(f"✅ Computed {len(curvs)} curvature values")
    if len(curvs) > 0:
        print(f"   Mean curvature: {curvs.mean():.6e}")
except Exception as e:
    print(f"❌ Error in spectral curvature: {e}")

# ============================================================================
# CELL 11: METRIC 2 - Prediction-based Thermodynamic Length
# ============================================================================
print("\n" + "="*80)
print("METRIC 2: THERMODYNAMIC LENGTH (PREDICTION-BASED)")
print("="*80)

@torch.no_grad()
def compute_prediction_thermodynamic_length(dataset, dataset_name: str):
    """Compute Fisher-Rao distance between consecutive layer predictions"""
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=causal_collate_standard
    )

    num_steps = num_layers - 1
    fr_step_sums = torch.zeros(num_steps, device=DEVICE)
    fr_step_counts = torch.zeros(num_steps, device=DEVICE)

    print(f"\n🔥 Computing for {dataset_name}...")

    for batch in tqdm(loader, desc=f"[{dataset_name}]"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        B, S = input_ids.shape

        # Get initial embeddings
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
            h = llama_model.embed_tokens(input_ids)

        logp_prev = None

        for ell in range(num_layers):
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
                layer_output = blocks[ell](h, attention_mask=attention_mask)
                h = layer_output[0]

                logits = lm_head(final_norm(h))
                logp = F.log_softmax(logits.float(), dim=-1)

            if logp_prev is not None:
                valid = (labels != -100)
                fr_dists = fisher_rao_distance(logp_prev, logp)

                step_idx = ell - 1
                fr_step_sums[step_idx] += fr_dists.masked_fill(~valid, 0.0).sum()
                fr_step_counts[step_idx] += valid.sum()

            logp_prev = logp

    fr_step_means = (fr_step_sums / fr_step_counts.clamp_min(1)).cpu().numpy()
    print(f"✅ Computed prediction-based thermodynamic length")
    return fr_step_means

# Compute for SQuAD
try:
    squad_pred_thermo = compute_prediction_thermodynamic_length(squad_dataset, "SQuAD v2")
except Exception as e:
    print(f"❌ Error: {e}")
    squad_pred_thermo = np.zeros(num_layers - 1)

# ============================================================================
# CELL 12: METRIC 3 - Belief Vector Fields
# ============================================================================
print("\n" + "="*80)
print("METRIC 3: BELIEF VECTOR FIELDS")
print("="*80)

@torch.no_grad()
def compute_belief_vectors(dataset, dataset_name: str):
    """Compute belief vector field magnitudes across layers"""
    loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        collate_fn=causal_collate_belief
    )

    v_sum = [torch.zeros(vocab_size, device=DEVICE, dtype=torch.float32) for _ in range(num_layers)]
    u_sum = [torch.zeros(vocab_size, device=DEVICE, dtype=torch.float32) for _ in range(num_layers)]
    cnt = [0 for _ in range(num_layers)]

    print(f"\n🔥 Computing for {dataset_name}...")

    for batch in tqdm(loader, desc=f"[{dataset_name}]"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        keep_mask = batch["select_mask"].to(DEVICE)

        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
                output_hidden_states=True
            )
            hidden_states = outputs.hidden_states

        for ell in range(num_layers):
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(DEVICE=="cuda")):
                h_ell = hidden_states[ell + 1]
                z = lm_head(final_norm(h_ell))

            keep = keep_mask.view(-1)
            if not keep.any():
                continue

            z_sel = z.view(-1, vocab_size)[keep]
            y_sel = labels.view(-1)[keep]

            z32 = z_sel.float() / TAU
            q = F.softmax(z32, dim=-1)
            q = torch.clamp(q, min=1e-12)
            u = torch.sqrt(q)

            g = -q / TAU
            add = torch.full((y_sel.shape[0], 1), 1.0/TAU, device=DEVICE, dtype=q.dtype)
            g.scatter_add_(dim=-1, index=y_sel.unsqueeze(-1), src=add)

            ug = u * g
            s = (q * g).sum(dim=-1, keepdim=True)
            t = (ug - s * u) / (2.0 * TAU)

            v_sum[ell] += t.sum(dim=0).to(v_sum[ell].dtype)
            u_sum[ell] += u.sum(dim=0).to(u_sum[ell].dtype)
            cnt[ell] += t.size(0)

    norms = []
    for ell in range(num_layers):
        if cnt[ell] == 0:
            norms.append(0.0)
            continue

        v_avg = v_sum[ell] / cnt[ell]
        u_avg = u_sum[ell] / cnt[ell]

        u_norm = torch.linalg.norm(u_avg).clamp_min(1e-12)
        u_bar = u_avg / u_norm

        radial = torch.dot(u_bar, v_avg)
        v_tan = v_avg - radial * u_bar

        n = torch.linalg.norm(v_tan).item()
        norms.append(2.0 * n if FR_NORM else n)

    print(f"✅ Computed belief vectors")
    return np.array(norms)

# Compute for SQuAD
try:
    squad_belief = compute_belief_vectors(squad_dataset, "SQuAD v2")
except Exception as e:
    print(f"❌ Error: {e}")
    squad_belief = np.zeros(num_layers)

# Compute for cultural data
cultural_belief_results = {}
try:
    for country in list(cultural_by_country.keys())[:2]:
        ds = cultural_by_country[country]
        if len(ds) > MAX_SAMPLES_CULTURAL:
            ds = ds.select(range(MAX_SAMPLES_CULTURAL))
        belief_norms = compute_belief_vectors(ds, f"Cultural-{country}")
        cultural_belief_results[country] = belief_norms
except Exception as e:
    print(f"❌ Error in cultural belief vectors: {e}")

# ============================================================================
# CELL 13: Visualization - All Metrics
# ============================================================================
print("\n" + "="*80)
print("GENERATING VISUALIZATIONS")
print("="*80)

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle('nDNA Analysis - Fine-tuned Llama Model', fontsize=16, fontweight='bold')

# Plot 1: Thermodynamic Length (Prediction-based)
ax = axes[0]
if len(squad_pred_thermo) > 0:
    steps = np.arange(1, len(squad_pred_thermo) + 1)
    ax.plot(steps, squad_pred_thermo, marker='^', color='blue', linewidth=2, label='SQuAD v2')
ax.set_title('Thermodynamic Length (Fisher-Rao Distance)', fontsize=13, fontweight='bold')
ax.set_xlabel('Layer Transition (ℓ → ℓ+1)', fontsize=11)
ax.set_ylabel('Mean FR Distance (radians)', fontsize=11)
ax.grid(True, alpha=0.3)
ax.legend()

# Plot 2: Belief Vectors - SQuAD
ax = axes[1]
if len(squad_belief) > 0:
    layers = np.arange(1, len(squad_belief) + 1)
    ax.plot(layers, squad_belief, marker='o', color='blue', linewidth=2, label='SQuAD v2')
ax.set_title('Belief Vector Field Magnitudes', fontsize=13, fontweight='bold')
ax.set_xlabel('Layer Index', fontsize=11)
ax.set_ylabel('||v_ℓ|| (Fisher-Rao norm)', fontsize=11)
ax.grid(True, alpha=0.3)
ax.legend()

# Plot 3: Belief Vectors - Cultural (if available)
ax = axes[2]
colors = ['green', 'orange', 'purple']
if cultural_belief_results:
    for i, (country, norms) in enumerate(cultural_belief_results.items()):
        layers = np.arange(1, len(norms) + 1)
        ax.plot(layers, norms, marker='s', color=colors[i % len(colors)],
                linewidth=2, label=country, markersize=6)
ax.set_title('Belief Vectors - CulturalBench', fontsize=13, fontweight='bold')
ax.set_xlabel('Layer Index', fontsize=11)
ax.set_ylabel('||v_ℓ|| (Fisher-Rao norm)', fontsize=11)
ax.grid(True, alpha=0.3)
if cultural_belief_results:
    ax.legend()

plt.tight_layout()
plt.show()

print("✅ Visualizations complete!")

# ============================================================================
# CELL 14: Summary Statistics
# ============================================================================
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print("\n📏 THERMODYNAMIC LENGTH (Prediction-based)")
print("-" * 60)
if len(squad_pred_thermo) > 0:
    print(f"SQuAD v2:")
    print(f"  Total FR distance: {squad_pred_thermo.sum():.6e} radians")
    print(f"  Mean per step: {squad_pred_thermo.mean():.6e} radians")
    print(f"  Max step: {squad_pred_thermo.max():.6e}")

print("\n🎯 BELIEF VECTORS")
print("-" * 60)
if len(squad_belief) > 0:
    print(f"SQuAD v2:")
    print(f"  Mean magnitude: {squad_belief.mean():.6e}")
    print(f"  Max magnitude: {squad_belief.max():.6e}")
    print(f"  Std deviation: {squad_belief.std():.6e}")

if cultural_belief_results:
    print(f"\nCulturalBench:")
    for country, norms in cultural_belief_results.items():
        print(f"  {country}: mean={norms.mean():.6e}, max={norms.max():.6e}")

print("\n" + "="*80)
print("✅ nDNA ANALYSIS COMPLETE!")
print("="*80)

In [ ]:
# Update this path to match your unzipped folder
MODEL_PATH = "/content/llama-3.2-3b-finetuned-CulturalBench"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Model path: {MODEL_PATH}")
print(f"Device: {DEVICE}\n")

In [ ]:
# ============================================================================
# CELL 1: Install Dependencies (Clean Reinstall for Compatibility)
# ============================================================================
%%capture
# Uninstall all potentially conflicting libraries
!pip uninstall -y \
    numpy \
    scipy \
    scikit-learn \
    pandas \
    seaborn \
    torch \
    torchvision \
    torchaudio \
    transformers \
    datasets \
    accelerate \
    peft \
    matplotlib \
    plotly \
    bitsandbytes

# Install core dependencies first in a specific order
!pip install -q -U \
    numpy==1.24.3 \
    scipy==1.11.4 \
    scikit-learn==1.3.2 \
    pandas==1.5.3 \
    seaborn==0.12.2

# Install other libraries
!pip install -q -U \
    torch==2.1.0 \
    torchvision==0.16.0 \
    torchaudio==2.1.0 \
    transformers==4.36.2 \
    datasets==2.16.1 \
    accelerate==0.25.0 \
    peft==0.7.1 \
    matplotlib==3.8.2 \
    plotly==5.18.0 \
    bitsandbytes==0.41.3 \
    --index-url https://download.pytorch.org/whl/cu118

print("✅ All dependencies installed successfully!")

# ============================================================================
# CELL 2: Imports
# ============================================================================
import torch
import torch.nn.functional as F
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel, LoraConfig
from tqdm import tqdm
import warnings
import gc
warnings.filterwarnings('ignore')

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔧 Device: {device}")
print(f"🔥 PyTorch version: {torch.__version__}")

if torch.cuda.is_available():
    print(f"🖥️  GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    torch.cuda.empty_cache()
    gc.collect()

# ============================================================================
# CELL 3: Load FINE-TUNED Model (Critical Fix!)
# ============================================================================
print("\n" + "="*80)
print("🤖 LOADING CULTURALLY FINE-TUNED LLAMA MODEL")
print("="*80)

# Configuration
BASE_MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
FINETUNED_ADAPTER_PATH = "./llama-3.2-3b-cultural-sft"  # Change to your path

# 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"📦 Loading base model: {BASE_MODEL_NAME}")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    output_hidden_states=True,
    trust_remote_code=True
)

# ⚠️ CRITICAL: Check if fine-tuned adapters exist
import os
if os.path.exists(FINETUNED_ADAPTER_PATH):
    print(f"✅ Loading fine-tuned LoRA adapters from: {FINETUNED_ADAPTER_PATH}")
    model = PeftModel.from_pretrained(base_model, FINETUNED_ADAPTER_PATH)
    print("✅ Fine-tuned model loaded successfully!")
else:
    print(f"⚠️  Warning: Fine-tuned adapters not found at {FINETUNED_ADAPTER_PATH}")
    print("⚠️  Using base model instead. Please run fine-tuning first!")
    model = base_model

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model.eval()
print(f"✅ Model ready!")
print(f"   Layers: {model.config.num_hidden_layers}")
print(f"   Hidden size: {model.config.hidden_size}")

# ============================================================================
# CELL 4: Load Datasets (CORRECTED Format Handling)
# ============================================================================
print("\n" + "="*80)
print("📚 LOADING DATASETS")
print("="*80)

# Load CulturalBench with proper error handling
print("Loading CulturalBench...")
try:
    cultural_dataset = load_dataset("kellycyy/CulturalBench", split="test")
    print(f"✅ CulturalBench loaded: {len(cultural_dataset)} examples")

    # Inspect actual structure
    print("\n📋 CulturalBench structure:")
    print(f"   Columns: {cultural_dataset.column_names}")
    print(f"\n   Sample example:")
    sample = cultural_dataset[0]
    for key, value in sample.items():
        print(f"   - {key}: {str(value)[:100]}...")

except Exception as e:
    print(f"❌ Error loading CulturalBench: {e}")
    print("⚠️  Creating fallback cultural dataset...")
    cultural_dataset = None

# Load SQuAD v2
print("\nLoading SQuAD v2...")
squad_dataset = load_dataset("rajpurkar/squad_v2", split="validation")
print(f"✅ SQuAD v2: {len(squad_dataset)} examples")

# Sample for T4 memory constraints
MAX_SAMPLES = 50  # Reduced for stability

if cultural_dataset:
    cultural_sample = cultural_dataset.shuffle(seed=42).select(range(min(MAX_SAMPLES, len(cultural_dataset))))
else:
    # Create minimal cultural dataset if loading fails
    cultural_sample = None

squad_sample = squad_dataset.shuffle(seed=42).select(range(min(MAX_SAMPLES, len(squad_dataset))))

print(f"\n📊 Analysis samples:")
print(f"   CulturalBench: {len(cultural_sample) if cultural_sample else 0}")
print(f"   SQuAD: {len(squad_sample)}")

# ============================================================================
# CELL 5: nDNA Metrics Implementation (Layer-wise Correct)
# ============================================================================
print("\n" + "="*80)
print("🔬 IMPLEMENTING nDNA METRICS (Corrected)")
print("="*80)

class nDNAMetrics:
    """
    Corrected nDNA metrics implementation for fine-tuned models
    Reference: https://pragyaai.github.io/ndna/llm/ndna/
    """

    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

        # Get actual number of layers from model
        if hasattr(model, 'model'):
            # For PEFT models
            if hasattr(model.model, 'model'):
                self.num_layers = len(model.model.model.layers)
            else:
                self.num_layers = model.config.num_hidden_layers
        else:
            self.num_layers = model.config.num_hidden_layers

        print(f"✅ Initialized with {self.num_layers} layers")

    def extract_hidden_states(self, text, max_length=256):
        """Extract hidden states with proper memory management"""
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model(
                **inputs,
                output_hidden_states=True,
                return_dict=True
            )

            # Access hidden states correctly for PEFT models
            if hasattr(outputs, 'hidden_states'):
                hidden_states = outputs.hidden_states
            else:
                raise ValueError("Model does not output hidden states")

            logits = outputs.logits
            attention_mask = inputs.attention_mask

        return hidden_states, logits, attention_mask

    def compute_spectral_curvature(self, hidden_states, attention_mask):
        """
        Spectral Curvature: κ_l = ||∇²h_l|| / (||∇h_l|| + ε)
        Measures information geometry curvature per layer
        """
        curvatures = []
        epsilon = 1e-8

        for i in range(len(hidden_states) - 1):
            h_curr = hidden_states[i].float()  # (batch, seq, hidden)
            h_next = hidden_states[i + 1].float()

            # First derivative approximation
            delta_h = h_next - h_curr

            # Second derivative approximation
            if i < len(hidden_states) - 2:
                h_next_next = hidden_states[i + 2].float()
                delta_h_next = delta_h_next - delta_h
            else:
                second_deriv = delta_h

            # Apply attention mask
            mask = attention_mask.unsqueeze(-1).float()
            delta_h_masked = delta_h * mask
            second_deriv_masked = second_deriv * mask

            # Compute norms (sum over sequence and hidden dimensions)
            grad_norm = torch.sqrt((delta_h_masked ** 2).sum()) + epsilon
            curv_norm = torch.sqrt((second_deriv_masked ** 2).sum())

            # Spectral curvature
            kappa = (curv_norm / grad_norm).item()
            curvatures.append(kappa)

        return np.array(curvatures)

    def compute_thermodynamic_length(self, hidden_states, attention_mask):
        """
        Thermodynamic Length: L_l = √(g_ij * dx^i * dx^j)
        Fisher information-based distance through layer space
        """
        thermo_lengths = []
        epsilon = 1e-8

        for i in range(len(hidden_states) - 1):
            h_curr = hidden_states[i].float()
            h_next = hidden_states[i + 1].float()

            # State difference (tangent vector)
            delta_h = h_next - h_curr

            # Apply mask
            mask = attention_mask.unsqueeze(-1).float()
            delta_h_masked = delta_h * mask

            # Fisher information metric approximation: g_ij ≈ E[∇log p ∇log p^T]
            # Simplified: use squared norm of state change
            fisher_metric = (delta_h_masked ** 2).sum()

            # Thermodynamic length element
            length = torch.sqrt(fisher_metric + epsilon).item()
            thermo_lengths.append(length)

        return np.array(thermo_lengths)

    def compute_belief_vector_entropy(self, logits, attention_mask):
        """
        Belief Vector Entropy: H[p(x|h)] = -Σ p(x) log p(x)
        Measures uncertainty in output distribution
        """
        # Apply softmax to get probabilities
        probs = F.softmax(logits, dim=-1)  # (batch, seq, vocab)

        # Compute entropy per position
        epsilon = 1e-10
        log_probs = torch.log(probs + epsilon)
        entropy = -(probs * log_probs).sum(dim=-1)  # (batch, seq)

        # Apply mask and average
        mask = attention_mask.float()
        masked_entropy = (entropy * mask).sum() / mask.sum()

        return masked_entropy.item()

    def compute_belief_divergence(self, logits, attention_mask):
        """
        KL divergence between consecutive token distributions
        D_KL(p_t || p_{t+1})
        """
        probs = F.softmax(logits, dim=-1)  # (batch, seq, vocab)

        divergences = []
        epsilon = 1e-10

        for t in range(probs.shape[1] - 1):
            p_curr = probs[:, t, :] + epsilon
            p_next = probs[:, t + 1, :] + epsilon

            # Normalize
            p_curr = p_curr / p_curr.sum(dim=-1, keepdim=True)
            p_next = p_next / p_next.sum(dim=-1, keepdim=True)

            # KL divergence
            kl = (p_curr * (torch.log(p_curr) - torch.log(p_next))).sum(dim=-1)
            divergences.append(kl.mean().item())

        return np.array(divergences)

    def analyze_text(self, text, max_length=256):
        """Complete nDNA analysis pipeline"""
        try:
            # Extract representations
            hidden_states, logits, attention_mask = self.extract_hidden_states(text, max_length)

            # Compute metrics
            spectral_curvature = self.compute_spectral_curvature(hidden_states, attention_mask)
            thermo_length = self.compute_thermodynamic_length(hidden_states, attention_mask)
            belief_entropy = self.compute_belief_vector_entropy(logits, attention_mask)
            belief_div = self.compute_belief_divergence(logits, attention_mask)

            # Clear GPU memory
            del hidden_states, logits
            torch.cuda.empty_cache()

            return {
                'spectral_curvature': spectral_curvature,
                'thermodynamic_length': thermo_length,
                'belief_entropy': belief_entropy,
                'belief_divergence': belief_div,
                'num_layers': len(spectral_curvature)
            }

        except Exception as e:
            print(f"⚠️ Error analyzing text: {e}")
            return None

# Initialize metrics computer
ndna = nDNAMetrics(model, tokenizer, device)
print("✅ nDNA Metrics initialized!")

# ============================================================================
# CELL 6: Prepare Text Functions (CORRECTED for Real Datasets)
# ============================================================================
def prepare_text_from_cultural(example):
    """Format CulturalBench - handle actual structure"""
    if 'question' in example:
        text = f"Question: {example['question']}"
        if 'choices' in example and example['choices']:
            choices_text = " ".join([f"({i+1}) {c}" for i, c in enumerate(example['choices'])])
            text += f"\nChoices: {choices_text}"
        return text
    elif 'text' in example:
        return example['text'][:500]
    else:
        # Fallback: use first text field found
        for key, value in example.items():
            if isinstance(value, str) and len(value) > 20:
                return value[:500]
        return "Cultural knowledge question"

def prepare_text_from_squad(example):
    """Format SQuAD with length limits"""
    context = example['context'][:300]  # Limit context
    question = example['question']
    return f"Context: {context}\nQuestion: {question}"

# ============================================================================
# CELL 7: Analyze Datasets (Memory-Efficient)
# ============================================================================
print("\n" + "="*80)
print("🔍 ANALYZING DATASETS WITH nDNA METRICS")
print("="*80)

def analyze_dataset_batch(dataset, prepare_fn, name, batch_size=5):
    """Analyze dataset in batches to avoid OOM"""
    results = {
        'spectral_curvature': [],
        'thermodynamic_length': [],
        'belief_entropy': [],
        'belief_divergence': []
    }

    num_samples = min(len(dataset), 30)  # Limit for T4

    for i in tqdm(range(0, num_samples, batch_size), desc=f"Analyzing {name}"):
        batch_end = min(i + batch_size, num_samples)

        for j in range(i, batch_end):
            try:
                example = dataset[j]
                text = prepare_fn(example)

                result = ndna.analyze_text(text, max_length=256)

                if result:
                    results['spectral_curvature'].append(result['spectral_curvature'])
                    results['thermodynamic_length'].append(result['thermodynamic_length'])
                    results['belief_entropy'].append(result['belief_entropy'])
                    results['belief_divergence'].append(result['belief_divergence'])

            except Exception as e:
                print(f"⚠️ Error on sample {j}: {e}")
                continue

        # Clear memory after each batch
        torch.cuda.empty_cache()
        gc.collect()

    return results

# Analyze CulturalBench
if cultural_sample:
    print("\n📊 Analyzing CulturalBench...")
    cultural_results = analyze_dataset_batch(
        cultural_sample,
        prepare_text_from_cultural,
        "CulturalBench"
    )
else:
    print("⚠️ Skipping CulturalBench analysis (dataset not loaded)")
    cultural_results = None

# Analyze SQuAD
print("\n📊 Analyzing SQuAD...")
squad_results = analyze_dataset_batch(
    squad_sample,
    prepare_text_from_squad,
    "SQuAD"
)

print(f"\n✅ Analysis complete!")
if cultural_results:
    print(f"   CulturalBench: {len(cultural_results['spectral_curvature'])} samples")
print(f"   SQuAD: {len(squad_results['spectral_curvature'])} samples")

# ============================================================================
# CELL 8: Aggregate Results (Safe Averaging)
# ============================================================================
print("\n" + "="*80)
print("📈 AGGREGATING RESULTS")
print("="*80)

def safe_aggregate(results):
    """Safely aggregate results with error handling"""
    if not results or not results['spectral_curvature']:
        return None

    return {
        'spectral_curvature': np.mean(results['spectral_curvature'], axis=0),
        'thermodynamic_length': np.mean(results['thermodynamic_length'], axis=0),
        'belief_entropy_mean': np.mean(results['belief_entropy']),
        'belief_divergence_mean': np.mean([np.mean(bd) for bd in results['belief_divergence'] if len(bd) > 0])
    }

cultural_avg = safe_aggregate(cultural_results) if cultural_results else None
squad_avg = safe_aggregate(squad_results)

if cultural_avg:
    print(f"✅ CulturalBench averages:")
    print(f"   Spectral curvature: {cultural_avg['spectral_curvature'].mean():.6f}")
    print(f"   Thermodynamic length: {cultural_avg['thermodynamic_length'].sum():.6f}")
    print(f"   Belief entropy: {cultural_avg['belief_entropy_mean']:.6f}")

print(f"\n✅ SQuAD averages:")
print(f"   Spectral curvature: {squad_avg['spectral_curvature'].mean():.6f}")
print(f"   thermodynamic_length': {squad_avg['thermodynamic_length'].sum():.6f}")
print(f"   Belief entropy: {squad_avg['belief_entropy_mean']:.6f}")

# ============================================================================
# CELL 9: 3D Visualizations (Enhanced & Corrected)
# ============================================================================
print("\n" + "="*80)
print("🎨 CREATING 3D VISUALIZATIONS")
print("="*80)

num_layers = len(squad_avg['spectral_curvature'])
layer_indices = np.arange(num_layers)

# ========== PLOT 1: 3D Spectral Curvature Comparison ==========
fig1 = go.Figure()

# SQuAD trace (always available)
fig1.add_trace(go.Scatter3d(
    x=layer_indices,
    y=np.zeros(num_layers),
    z=squad_avg['spectral_curvature'],
    mode='lines+markers',
    name='SQuAD (Baseline)',
    line=dict(color='red', width=6),
    marker=dict(size=8, color='red'),
    hovertemplate='<b>SQuAD</b><br>Layer: %{x}<br>Curvature: %{z:.6f}<extra></extra>'
))

# CulturalBench trace (if available)
if cultural_avg:
    fig1.add_trace(go.Scatter3d(
        x=layer_indices,
        y=np.ones(num_layers),
        z=cultural_avg['spectral_curvature'],
        mode='lines+markers',
        name='CulturalBench (Fine-tuned)',
        line=dict(color='blue', width=6),
        marker=dict(size=8, color='blue'),
        hovertemplate='<b>CulturalBench</b><br>Layer: %{x}<br>Curvature: %{z:.6f}<extra></extra>'
    ))

    # Connecting lines
    for i in range(num_layers):
        fig1.add_trace(go.Scatter3d(
            x=[i, i],
            y=[0, 1],
            z=[squad_avg['spectral_curvature'][i], cultural_avg['spectral_curvature'][i]],
            mode='lines',
            line=dict(color='gray', width=2, dash='dot'),
            showlegend=False,
            hoverinfo='skip'
        ))

fig1.update_layout(
    title=dict(
        text='<b>nDNA Spectral Curvature: Fine-tuned vs Baseline</b><br>' +
             '<sub>Information Geometry Across Layers (Fine-tuned Model)</sub>',
        x=0.5,
        font=dict(size=18)
    ),
    scene=dict(
        xaxis_title='Layer Index',
        yaxis_title='Dataset (0=SQuAD, 1=Cultural)',
        zaxis_title='Spectral Curvature (κ)',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2)),
        bgcolor='rgba(240, 240, 240, 0.9)'
    ),
    width=1200,
    height=800,
    showlegend=True
)

fig1.show()

# ========== PLOT 2: Thermodynamic Length Trajectory ==========
squad_cumul = np.cumsum(squad_avg['thermodynamic_length'])

fig2 = go.Figure()

fig2.add_trace(go.Scatter3d(
    x=layer_indices,
    y=squad_avg['spectral_curvature'],
    z=squad_cumul,
    mode='lines+markers',
    name='SQuAD',
    line=dict(color='red', width=6),
    marker=dict(size=8, color=squad_cumul, colorscale='Reds', showscale=True,
                colorbar=dict(title='Cumulative<br>Length', x=1.1)),
    hovertemplate='<b>SQuAD</b><br>Layer: %{x}<br>Curvature: %{y:.6f}<br>Cumul Length: %{z:.6f}<extra></extra>'
))

if cultural_avg:
    cultural_cumul = np.cumsum(cultural_avg['thermodynamic_length'])
    fig2.add_trace(go.Scatter3d(
        x=layer_indices,
        y=cultural_avg['spectral_curvature'],
        z=cultural_cumul,
        mode='lines+markers',
        name='CulturalBench',
        line=dict(color='blue', width=6),
        marker=dict(size=8, color=cultural_cumul, colorscale='Blues', showscale=True,
                    colorbar=dict(title='Cumulative<br>Length', x=1.15)),
        hovertemplate='<b>CulturalBench</b><br>Layer: %{x}<br>Curvature: %{y:.6f}<br>Cumul Length: %{z:.6f}<extra></extra>'
    ))

fig2.update_layout(
    title=dict(
        text='<b>nDNA Thermodynamic Length: Fisher Information Trajectory</b><br>' +
             '<sub>Information-Geometric Path Through Layer Space</sub>',
        x=0.5,
        font=dict(size=18)
    ),
    scene=dict(
        xaxis_title='Layer Index',
        yaxis_title='Spectral Curvature (κ)',
        zaxis_title='Cumulative Thermodynamic Length (L)',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2)),
        bgcolor='rgba(240, 240, 240, 0.9)'
    ),
    width=1200,
    height=800,
    showlegend=True
)

fig2.show()

# ========== PLOT 3: Comprehensive Multi-Panel ==========
fig3 = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Spectral Curvature by Layer',
        'Thermodynamic Length by Layer',
        'Cumulative Thermodynamic Length',
        'Belief Entropy Comparison'
    ),
    specs=[
        [{'type': 'scatter'}, {'type': 'scatter'}],
        [{'type': 'scatter'}, {'type': 'bar'}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

# Plot 1: Spectral Curvature
fig3.add_trace(
    go.Scatter(x=layer_indices, y=squad_avg['spectral_curvature'],
               mode='lines+markers', name='SQuAD', line=dict(color='red', width=3),
               marker=dict(size=6)),
    row=1, col=1
)
if cultural_avg:
    fig3.add_trace(
        go.Scatter(x=layer_indices, y=cultural_avg['spectral_curvature'],
                   mode='lines+markers', name='Cultural', line=dict(color='blue', width=3),
                   marker=dict(size=6)),
        row=1, col=1
    )

# Plot 2: Thermodynamic Length
fig3.add_trace(
    go.Scatter(x=layer_indices, y=squad_avg['thermodynamic_length'],
               mode='lines+markers', name='SQuAD', line=dict(color='red', width=3),
               marker=dict(size=6), showlegend=False),
    row=1, col=2
)
if cultural_avg:
    fig3.add_trace(
        go.Scatter(x=layer_indices, y=cultural_avg['thermodynamic_length'],
                   mode='lines+markers', name='Cultural', line=dict(color='blue', width=3),
                   marker=dict(size=6), showlegend=False),
        row=1, col=2
    )

# Plot 3: Cumulative Length
fig3.add_trace(
    go.Scatter(x=layer_indices, y=squad_cumul,
               mode='lines+markers', name='SQuAD', line=dict(color='red', width=3),
               marker=dict(size=6), showlegend=False),
    row=2, col=1
)
if cultural_avg:
    fig3.add_trace(
        go.Scatter(x=layer_indices, y=cultural_cumul,
                   mode='lines+markers', name='Cultural', line=dict(color='blue', width=3),
                   marker=dict(size=6), showlegend=False),
        row=2, col=1
    )

# Plot 4: Belief Entropy
if cultural_avg:
    fig3.add_trace(
        go.Bar(x=['SQuAD', 'CulturalBench'],
               y=[squad_avg['belief_entropy_mean'], cultural_avg['belief_entropy_mean']],
               marker_color=['red', 'blue'],
               showlegend=False),
        row=2, col=2
    )
else:
    fig3.add_trace(
        go.Bar(x=['SQuAD'],
               y=[squad_avg['belief_entropy_mean']],
               marker_color=['red'],
               showlegend=False),
        row=2, col=2
    )

fig3.update_xaxes(title_text="Layer Index", row=1, col=1)
fig3.update_xaxes(title_text="Layer Index", row=1, col=2)
fig3.update_xaxes(title_text="Layer Index", row=2, col=1)
fig3.update_xaxes(title_text="Dataset", row=2, col=2)

fig3.update_yaxes(title_text="Curvature (κ)", row=1, col=1)
fig3.update_yaxes(title_text="Length (L)", row=1, col=2)
fig3.update_yaxes(title_text="Cumulative L", row=2, col=1)
fig3.update_yaxes(title_text="Entropy", row=2, col=2)

fig3.update_layout(
    title_text='<b>Complete nDNA Analysis: Fine-tuned Model Evaluation</b>',
    height=900,
    width=1400,
    showlegend=True
)

fig3.show()

print("\n✅ All visualizations created successfully!")

# ============================================================================
# CELL 10: Statistical Summary
# ============================================================================
print("\n" + "="*80)
print("📊 STATISTICAL SUMMARY")
print("="*80)

if cultural_avg:
    print(f"\n{'Metric':<30} {'CulturalBench':<20} {'SQuAD':<20} {'Difference':<15}")
    print("="*85)
    print(f"{'Spectral Curvature (mean)':<30} {cultural_avg['spectral_curvature'].mean():<20.6f} {squad_avg['spectral_curvature'].mean():<20.6f} {cultural_avg['spectral_curvature'].mean() - squad_avg['spectral_curvature'].mean():<15.6f}")
    print(f"{'Thermodynamic Length (sum)':<30} {cultural_avg['thermodynamic_length'].sum():<20.6f} {squad_avg['thermodynamic_length'].sum():<20.6f} {cultural_avg['thermodynamic_length'].sum() - squad_avg['thermodynamic_length'].sum():<15.6f}")
    print(f"{'Belief Entropy':<30} {cultural_avg['belief_entropy_mean']:<20.6f} {squad_avg['belief_entropy_mean']:<20.6f} {cultural_avg['belief_entropy_mean'] - squad_avg['belief_entropy_mean']:<15.6f}")
    print("="*85)

    print("\n📊 LAYER-WISE BREAKDOWN:")
    print(f"{'Layer':<8} {'Cultural κ':<15} {'SQuAD κ':<15} {'Cultural L':<15} {'SQuAD L':<15}")
    print("-" * 68)
    for i in range(num_layers):
        print(f"{i:<8} {cultural_avg['spectral_curvature'][i]:<15.6f} "
              f"{squad_avg['spectral_curvature'][i]:<15.6f} "
              f"{cultural_avg['thermodynamic_length'][i]:<15.6f} "
              f"{squad_avg['thermodynamic_length'][i]:<15.6f}")
else:
    print("⚠️ Only SQuAD analysis available")
    print(f"\nSQuAD Metrics:")
    print(f"   Spectral Curvature: {squad_avg['spectral_curvature'].mean():.6f}")
    print(f"   Thermodynamic Length: {squad_avg['thermodynamic_length'].sum():.6f}")
    print(f"   Belief Entropy: {squad_avg['belief_entropy_mean']:.6f}")

print("\n✅ ANALYSIS COMPLETE!")
print("="*80)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
#import seaborn as sns
from tqdm import tqdm
import warnings
import gc

In [ ]:
MODEL_PATH = "/content/llama-3.2-3b-finetuned-culturalbench"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_SAMPLES = 10  # Reduced for stability
MAX_SEQ_LEN = 128  # Reduced for T4 GPU
BATCH_SIZE = 1

In [ ]:
print(f"Device: {DEVICE}")
print(f"Model: {MODEL_PATH}")
print(f"Samples: {NUM_SAMPLES} per dataset")
print(f"Max sequence length: {MAX_SEQ_LEN}\n")

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

model = None
tokenizer = None
final_norm = None
lm_head = None

try:
    # Load tokenizer
    print("1. Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print(f"   ✓ Tokenizer loaded (vocab: {tokenizer.vocab_size})\n")

    # Try loading as PEFT model (if you used LoRA)
    print("2. Loading model...")
    try:
        print("   Attempting PEFT/LoRA model load...")
        base_model = AutoModelForCausalLM.from_pretrained(
            "meta-llama/Llama-3.2-3B-Instruct",
            torch_dtype=torch.float16,
            device_map="auto",
            low_cpu_mem_usage=True
        )
        model = PeftModel.from_pretrained(base_model, MODEL_PATH)
        model = model.merge_and_unload()
        print("   ✓ PEFT model loaded and merged\n")
    except Exception as e:
        print(f"   PEFT load failed: {str(e)[:50]}")
        print("   Attempting direct model load...")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            torch_dtype=torch.float16,
            device_map="auto",
            low_cpu_mem_usage=True
        )
        print("   ✓ Model loaded directly\n")

    model.eval()

    # Get model components
    print("3. Verifying model structure...")
    vocab_size = model.config.vocab_size
    num_layers = model.config.num_hidden_layers

    # Get final norm (layer norm before output)
    if hasattr(model, 'model') and hasattr(model.model, 'norm'):
        final_norm = model.model.norm
        print(f"   ✓ Final layer norm: {type(final_norm).__name__}")
    else:
        final_norm = None
        print(f"   ⚠ No final norm (will use identity)")

    # Get LM head
    if hasattr(model, 'lm_head'):
        lm_head = model.lm_head
        print(f"   ✓ LM head: {type(lm_head).__name__}")
    else:
        raise AttributeError("Model must have lm_head")

    print(f"   ✓ Layers: {num_layers}")
    print(f"   ✓ Vocab: {vocab_size}\n")

except Exception as e:
    print(f"\n❌ CRITICAL ERROR loading model: {e}")
    print("\nPlease verify:")
    print("1. Model path is correct")
    print("2. Model files exist (config.json, pytorch_model.bin)")
    print("3. Model is properly saved\n")
    raise

# ============================================================================
# STEP 2: LOAD DATASETS
# ============================================================================

print("="*80)
print("LOADING DATASETS")
print("="*80 + "\n")

from datasets import load_dataset

# SQuAD v2
squad_texts = []
try:
    print("Loading SQuAD v2...")
    squad_data = load_dataset("rajpurkar/squad_v2", split=f"validation[:{NUM_SAMPLES*2}]")
    for sample in squad_data:
        context = sample["context"].strip()
        if 50 < len(context) < 500:
            squad_texts.append(context)
            if len(squad_texts) >= NUM_SAMPLES:
                break
    print(f"✓ Loaded {len(squad_texts)} SQuAD samples\n")
except Exception as e:
    print(f"⚠ SQuAD error: {str(e)[:50]}")
    squad_texts = [
        "Machine learning enables systems to learn from data without explicit programming.",
        "Neural networks consist of layers that process information hierarchically.",
        "Deep learning has revolutionized computer vision and natural language processing."
    ]
    print(f"✓ Using {len(squad_texts)} fallback texts\n")

# CulturalBench
cultural_texts = []
try:
    print("Loading CulturalBench...")
    cultural_data = load_dataset("kellycyy/CulturalBench", "CulturalBench-Hard", split=f"test[:{NUM_SAMPLES*2}]")
    for sample in cultural_data:
        try:
            q = str(sample.get("prompt_question", "")).strip()
            a = str(sample.get("answer", "")).strip()
            if q and a:
                text = f"{q} {a}".strip()
                if 50 < len(text) < 500:
                    cultural_texts.append(text)
                    if len(cultural_texts) >= NUM_SAMPLES:
                        break
        except:
            continue
    print(f"✓ Loaded {len(cultural_texts)} CulturalBench samples\n")
except Exception as e:
    print(f"⚠ CulturalBench error: {str(e)[:50]}")
    cultural_texts = [
        "Cultural diversity enriches global communication and understanding.",
        "Traditional knowledge systems provide unique perspectives on sustainability.",
        "Cross-cultural competence requires empathy and active listening skills."
    ]
    print(f"✓ Using {len(cultural_texts)} fallback texts\n")

print(f"✓ Total: {len(squad_texts) + len(cultural_texts)} samples\n")

# ============================================================================
# OFFICIAL nDNA FORMULAS (VERIFIED)
# ============================================================================

def compute_spectral_curvature(hidden_states):
    """
    OFFICIAL FORMULA: κ_ℓ = ||Δ²h_ℓ|| = ||h_{ℓ+1} - 2h_ℓ + h_{ℓ-1}||

    Args:
        hidden_states: List of hidden state tensors [seq_len, hidden_dim]

    Returns:
        Array of curvature values (length = num_layers - 2)
    """
    kappas = []

    for ell in range(1, len(hidden_states) - 1):
        h_prev = hidden_states[ell - 1]
        h_curr = hidden_states[ell]
        h_next = hidden_states[ell + 1]

        # Second-order difference (discrete Laplacian)
        delta2_h = h_next - 2.0 * h_curr + h_prev

        # L2 norm (curvature)
        kappa = torch.norm(delta2_h, p=2).item()
        kappas.append(kappa)

    return np.array(kappas)

def compute_thermodynamic_length(logp_sequence):
    """
    OFFICIAL FORMULA: L_ℓ = Σ ||∇_θ log p_ℓ||²

    Approximation: L_ℓ ≈ Σ_i (log p_{ℓ+1,i} - log p_{ℓ,i})²

    CORRECTED: Sum over vocab FIRST, then total

    Args:
        logp_sequence: List of log-probability tensors [vocab_size]

    Returns:
        Array of thermodynamic length values
    """
    lengths = []

    for ell in range(len(logp_sequence) - 1):
        logp_curr = logp_sequence[ell]
        logp_next = logp_sequence[ell + 1]

        # Difference in log-space
        delta_logp = logp_next - logp_curr

        # CORRECTED: Sum of squared differences over vocab
        length = torch.sum(delta_logp ** 2).item()
        lengths.append(length)

    return np.array(lengths)

def compute_belief_vector(logp_sequence):
    """
    OFFICIAL FORMULA: ||v_ℓ^(c)|| = ||E[∇_{h_ℓ} log p]||

    Approximation: ||v_ℓ|| ≈ ||log p_{ℓ+1} - log p_ℓ||

    Args:
        logp_sequence: List of log-probability tensors [vocab_size]

    Returns:
        Array of belief vector magnitudes
    """
    beliefs = []

    for ell in range(len(logp_sequence) - 1):
        logp_curr = logp_sequence[ell]
        logp_next = logp_sequence[ell + 1]

        # Belief gradient
        delta_theta = logp_next - logp_curr

        # L2 magnitude
        belief_mag = torch.norm(delta_theta, p=2).item()
        beliefs.append(belief_mag)

    return np.array(beliefs)

# ============================================================================
# MAIN PROCESSING LOOP
# ============================================================================

print("="*80)
print("COMPUTING nDNA METRICS")
print("="*80 + "\n")

# Storage
results = {
    'squad': {'kappas': [], 'lengths': [], 'beliefs': []},
    'cultural': {'kappas': [], 'lengths': [], 'beliefs': []}
}

model.eval()

def process_text(text, dataset_name):
    """Process single text and extract metrics"""
    try:
        # Tokenize
        enc = tokenizer(
            text,
            return_tensors="pt",
            max_length=MAX_SEQ_LEN,
            truncation=True,
            padding=True
        ).to(DEVICE)

        if enc['input_ids'].shape[1] < 5:
            return None

        # Forward pass
        with torch.no_grad():
            outputs = model(
                enc['input_ids'],
                attention_mask=enc.get('attention_mask'),
                output_hidden_states=True
            )

        hidden_states = outputs.hidden_states

        if not hidden_states or len(hidden_states) < 3:
            return None

        # Extract last token hidden states at each layer
        h_list = []
        logp_list = []

        seq_len = enc['input_ids'].shape[1]

        for h_layer in hidden_states:
            # Get last token
            h_last = h_layer[0, seq_len - 1, :]  # [hidden_dim]

            # Apply final norm if available (keep on GPU)
            if final_norm is not None:
                h_norm = final_norm(h_last)
            else:
                h_norm = h_last

            # Get logits and log-probs
            logits = lm_head(h_norm)
            logp = F.log_softmax(logits, dim=-1)

            # Store (move to CPU for memory)
            h_list.append(h_last.cpu())
            logp_list.append(logp.cpu())

        # Compute metrics
        kappas = compute_spectral_curvature(h_list)
        lengths = compute_thermodynamic_length(logp_list)
        beliefs = compute_belief_vector(logp_list)

        # Cleanup
        del outputs, hidden_states, enc, h_list, logp_list
        torch.cuda.empty_cache()

        return {
            'kappas': kappas,
            'lengths': lengths,
            'beliefs': beliefs
        }

    except Exception as e:
        print(f"⚠ Error: {str(e)[:50]}")
        return None

# Process SQuAD
print("Processing SQuAD v2...")
for text in tqdm(squad_texts):
    metrics = process_text(text, "squad")
    if metrics:
        results['squad']['kappas'].append(metrics['kappas'])
        results['squad']['lengths'].append(metrics['lengths'])
        results['squad']['beliefs'].append(metrics['beliefs'])
    gc.collect()

print()

# Process CulturalBench
print("Processing CulturalBench...")
for text in tqdm(cultural_texts):
    metrics = process_text(text, "cultural")
    if metrics:
        results['cultural']['kappas'].append(metrics['kappas'])
        results['cultural']['lengths'].append(metrics['lengths'])
        results['cultural']['beliefs'].append(metrics['beliefs'])
    gc.collect()

print("\n✓ Metrics computed\n")

# ============================================================================
# AGGREGATE RESULTS
# ============================================================================

def aggregate_metrics(metric_list):
    """Aggregate metrics across samples"""
    if not metric_list:
        return None

    # Find max length
    max_len = max(len(m) for m in metric_list)

    # Pad and average
    padded = np.zeros((len(metric_list), max_len))
    for i, m in enumerate(metric_list):
        padded[i, :len(m)] = m

    mean_metric = np.mean(padded, axis=0)
    std_metric = np.std(padded, axis=0)

    return {'mean': mean_metric, 'std': std_metric, 'raw': padded}

# Aggregate
agg = {}
for dataset in ['squad', 'cultural']:
    agg[dataset] = {}
    for metric in ['kappas', 'lengths', 'beliefs']:
        agg[dataset][metric] = aggregate_metrics(results[dataset][metric])

print(f"Squad samples: {len(results['squad']['kappas'])}")
print(f"Cultural samples: {len(results['cultural']['kappas'])}")
print(f"Layers analyzed: {len(agg['squad']['kappas']['mean']) if agg['squad']['kappas'] else 0}\n")

# ============================================================================
# VISUALIZATION 1: TRUE 3D SURFACE PLOTS
# ============================================================================

print("Creating visualizations...\n")

if agg['squad']['kappas'] and agg['cultural']['kappas']:

    fig = plt.figure(figsize=(18, 5))

    for idx, (dataset, title, cmap) in enumerate([
        ('squad', 'SQuAD v2', 'viridis'),
        ('cultural', 'CulturalBench', 'plasma')
    ]):

        raw_data = agg[dataset]['kappas']['raw']
        n_samples, n_layers = raw_data.shape

        # Create TRUE 3D surface
        X, Y = np.meshgrid(np.arange(n_layers), np.arange(n_samples))
        Z = raw_data

        ax = fig.add_subplot(1, 3, idx + 1, projection='3d')
        surf = ax.plot_surface(X, Y, Z, cmap=cmap, alpha=0.8, edgecolor='none')

        ax.set_xlabel('Layer', fontsize=10)
        ax.set_ylabel('Sample', fontsize=10)
        ax.set_zlabel('Spectral Curvature κ', fontsize=10)
        ax.set_title(f'Spectral Curvature: {title}', fontweight='bold')
        ax.view_init(elev=20, azim=45)
        fig.colorbar(surf, ax=ax, shrink=0.5)

    # Comparison plot
    ax = fig.add_subplot(1, 3, 3)
    layers_sq = np.arange(len(agg['squad']['kappas']['mean']))
    layers_cu = np.arange(len(agg['cultural']['kappas']['mean']))

    ax.plot(layers_sq, agg['squad']['kappas']['mean'], 'o-', label='SQuAD v2',
            linewidth=2, markersize=5, color='#2ecc71')
    ax.fill_between(layers_sq,
                     agg['squad']['kappas']['mean'] - agg['squad']['kappas']['std'],
                     agg['squad']['kappas']['mean'] + agg['squad']['kappas']['std'],
                     alpha=0.2, color='#2ecc71')

    ax.plot(layers_cu, agg['cultural']['kappas']['mean'], 's-', label='CulturalBench',
            linewidth=2, markersize=5, color='#e74c3c')
    ax.fill_between(layers_cu,
                     agg['cultural']['kappas']['mean'] - agg['cultural']['kappas']['std'],
                     agg['cultural']['kappas']['mean'] + agg['cultural']['kappas']['std'],
                     alpha=0.2, color='#e74c3c')

    ax.set_xlabel('Layer', fontweight='bold')
    ax.set_ylabel('Spectral Curvature κ', fontweight='bold')
    ax.set_title('Comparison (Mean ± Std)', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('nDNA_spectral_curvature.png', dpi=200, bbox_inches='tight')
    plt.show()
    print("✓ Saved: nDNA_spectral_curvature.png\n")

# ============================================================================
# VISUALIZATION 2: THERMODYNAMIC LENGTH
# ============================================================================

if agg['squad']['lengths'] and agg['cultural']['lengths']:

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for idx, (dataset, title, color) in enumerate([
        ('squad', 'SQuAD v2', '#3498db'),
        ('cultural', 'CulturalBench', '#e67e22')
    ]):

        ax = axes[idx]
        layers = np.arange(len(agg[dataset]['lengths']['mean']))
        mean = agg[dataset]['lengths']['mean']
        std = agg[dataset]['lengths']['std']

        ax.plot(layers, mean, 'o-', linewidth=2.5, markersize=6, color=color)
        ax.fill_between(layers, mean - std, mean + std, alpha=0.2, color=color)

        ax.set_xlabel('Layer Transition', fontweight='bold')
        ax.set_ylabel('Thermodynamic Length L', fontweight='bold')
        ax.set_title(f'Thermodynamic Length: {title}', fontweight='bold')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('nDNA_thermodynamic_length.png', dpi=200, bbox_inches='tight')
    plt.show()
    print("✓ Saved: nDNA_thermodynamic_length.png\n")

# ============================================================================
# VISUALIZATION 3: BELIEF VECTOR
# ============================================================================

if agg['squad']['beliefs'] and agg['cultural']['beliefs']:

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for idx, (dataset, title, color) in enumerate([
        ('squad', 'SQuAD v2', '#9b59b6'),
        ('cultural', 'CulturalBench', '#16a085')
    ]):

        ax = axes[idx]
        layers = np.arange(len(agg[dataset]['beliefs']['mean']))
        mean = agg[dataset]['beliefs']['mean']
        std = agg[dataset]['beliefs']['std']

        ax.plot(layers, mean, '^-', linewidth=2.5, markersize=6, color=color)
        ax.fill_between(layers, mean - std, mean + std, alpha=0.2, color=color)

        # Add mean line
        ax.axhline(np.mean(mean), linestyle=':', linewidth=2, color=color, alpha=0.5,
                   label=f'Mean={np.mean(mean):.4f}')

        ax.set_xlabel('Layer Transition', fontweight='bold')
        ax.set_ylabel('Belief Vector ||v||', fontweight='bold')
        ax.set_title(f'Belief Vector: {title}', fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('nDNA_belief_vector.png', dpi=200, bbox_inches='tight')
    plt.show()
    print("✓ Saved: nDNA_belief_vector.png\n")

# ============================================================================
# SUMMARY STATISTICS
# ============================================================================

print("="*80)
print("nDNA ANALYSIS SUMMARY")
print("="*80 + "\n")

for dataset, name in [('squad', 'SQuAD v2'), ('cultural', 'CulturalBench')]:
    print(f"{name}:")
    print(f"  Samples: {len(results[dataset]['kappas'])}")

    for metric, label in [('kappas', 'Spectral Curvature'),
                          ('lengths', 'Thermodynamic Length'),
                          ('beliefs', 'Belief Vector')]:
        if agg[dataset][metric]:
            mean_val = np.mean(agg[dataset][metric]['mean'])
            std_val = np.mean(agg[dataset][metric]['std'])
            print(f"  {label}:")
            print(f"    Mean: {mean_val:.6f}")
            print(f"    Avg Std: {std_val:.6f}")
    print()

In [ ]:
import subprocess
import sys

print("Fixing numpy/pyarrow compatibility...\n")

# Clear pip cache
subprocess.check_call([sys.executable, "-m", "pip", "cache", "purge"])

# Install compatible versions
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy==1.23.5"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow==10.0.1"])

print("✓ Dependencies fixed!")
print("\n⚠ NEXT: Click Runtime > Restart runtime")

In [ ]:
# Verify fix
print("Testing imports...")
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import numpy, pyarrow, torch
    print(f"✓ numpy: {numpy.__version__}")
    print(f"✓ pyarrow: {pyarrow.__version__}")
    print("✅ ALL FIXED! Run nDNA code [108] now.")
except Exception as e:
    print(f"❌ Still broken: {e}")


In [ ]:
# Cell 1: Install Core Dependencies (T4-Safe)
!pip install -q transformers datasets torch plotly numpy accelerate bitsandbytes
import warnings; warnings.filterwarnings('ignore')
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import gc
from torch.nn.functional import softmax, log_softmax

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Device: {device} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
!pip install -q transformers datasets torch plotly numpy accelerate bitsandbytes
import warnings; warnings.filterwarnings('ignore')
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import gc
from torch.nn.functional import softmax, log_softmax

In [ ]:
# Cell 2: Load Datasets (Subsets for T4 Efficiency)
print("📚 Loading Datasets...")
squad = load_dataset("squad_v2", split="train[:20]")  # 20 QA pairs
squad_texts = [f"Context: {item['context'][:200]}... Question: {item['question']}" for item in squad]  # Truncate for speed

cultural = load_dataset("kellycyy/CulturalBench", split="test[:20]")  # 20 cultural questions
cultural_texts = [item['question'] for item in cultural]  # Direct questions

all_texts = squad_texts + cultural_texts
labels = ['SQuAD'] * 20 + ['CulturalBench'] * 20
print(f"✅ Loaded: SQuAD(20), CulturalBench(20) | Total Texts: {len(all_texts)}")

In [ ]:
!pip install numpy==1.26.4 --force-reinstall -q
!pip install -q transformers datasets plotly torch accelerate bitsandbytes

In [ ]:
import numpy as np
import pandas as pd  # If used
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__ if 'pd' in locals() else 'N/A'}")

In [ ]:
# Cell 2: Load Datasets (Subsets for T4 Efficiency)
print("📚 Loading Datasets...")
squad = load_dataset("squad_v2", split="train[:20]")  # 20 QA pairs
squad_texts = [f"Context: {item['context'][:200]}... Question: {item['question']}" for item in squad]  # Truncate for speed

cultural = load_dataset("kellycyy/CulturalBench", split="test[:20]")  # 20 cultural questions
cultural_texts = [item['prompt_question'] for item in cultural]  # Direct questions

all_texts = squad_texts + cultural_texts
labels = ['SQuAD'] * 20 + ['CulturalBench'] * 20
print(f"✅ Loaded: SQuAD(20), CulturalBench(20) | Total Texts: {len(all_texts)}")

In [ ]:
# Cell 6: Intuitive Layer-wise Plots (3D Trajectories)
# 3D: Layer vs Curvature vs Thermo (Per Dataset)
fig_3d = go.Figure()
colors = {'SQuAD': 'blue', 'CulturalBench': 'red'}
for ds, m in data.items():
    fig_3d.add_trace(go.Scatter3d(
        x=layers, y=m['curvatures'], z=m['thermo_lengths'],
        mode='lines+markers', name=ds,
        line=dict(color=colors[ds], width=4),
        marker=dict(size=4, color=colors[ds]),
        hovertemplate=f'<b>{ds}</b><br>Layer: %{{x}}<br>κ_ℓ: %{{y:.4f}}<br>ℒ_ℓ: %{{z:.4f}}<extra></extra>'
    ))
fig_3d.update_layout(
    title="3D nDNA Trajectory: Spectral Curvature vs Thermodynamic Length (Layer-wise)",
    scene=dict(
        xaxis_title="Layer Depth (0-31: Embed to Output)",
        yaxis_title="Spectral Curvature κ_ℓ (Latent Bending)",
        zaxis_title="Thermodynamic Length ℒ_ℓ (Epistemic Effort)"
    ),
    height=600, showlegend=True
)
fig_3d.show()

# 3D: Layer vs Entropy vs Concentration (Belief Drift)
fig_belief = go.Figure()
for ds, m in data.items():
    fig_belief.add_trace(go.Scatter3d(
        x=layers, y=m['entropies'], z=m['concentrations'],
        mode='lines+markers', name=ds,
        line=dict(color=colors[ds], width=4),
        marker=dict(size=4, color=colors[ds]),
        hovertemplate=f'<b>{ds}</b><br>Layer: %{{x}}<br>Entropy: %{{y:.4f}}<br>Concentration: %{{z:.4f}}<extra></extra>'
    ))
fig_belief.update_layout(
    title="3D Belief Vector Field: Entropy vs Concentration (Cultural Drift)",
    scene=dict(
        xaxis_title="Layer Depth (0-31)",
        yaxis_title="Belief Entropy (Drift Diversity)",
        zaxis_title="Concentration (Focus Sharpness)"
    ),
    height=600
)
fig_belief.show()